# P2_G0 EDA 1단계: 고등교육통계 학교별X학과별 원자료 점검

목표는 원본 Excel에서 분석 가능한 DataFrame을 먼저 확정하는 것이다.

- 원본 시트/헤더 행 확인
- `df` 로드 및 타입 정리
- 컬럼 프로파일, 결측, 변환 실패 점검
- 수치형 기술통계
- 주요 범주형 분포와 후보 키 중복 점검


In [1]:

from pathlib import Path
import re
import warnings

from IPython.display import Markdown, display
from openpyxl import load_workbook
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=UserWarning)

pd.set_option("display.max_columns", 140)
pd.set_option("display.max_rows", 80)
pd.set_option("display.width", 220)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

BASE_DIR = Path("/home/sieg/projects-wsl/SBS_dataScience/workbook/p2/p2_2")
SOURCE_XLSX = BASE_DIR / "final/data/2024년 고등 학교별X학과별 입학정원 지원 입학 학생 외국인학생 졸업 교원_240912H.xlsx"
DATA_SHEET = "학교별 학과별 주요 현황"
HEADER_ROW_EXCEL = 14
DATA_START_ROW_EXCEL = HEADER_ROW_EXCEL + 1
OUTPUT_DIR = BASE_DIR / "final/eda/P2_G0"
TABLE_DIR = OUTPUT_DIR / "tables"
ANALYSIS_FRAME_DIR = OUTPUT_DIR / "analysis_frames"
FINAL_DATA_DIR = OUTPUT_DIR / "final_data"
OUTPUT_ENCODING = "utf-8-sig"
DROPPED_ANALYSIS_COLUMNS = ["시간강사_계", "시간강사_남", "시간강사_여"]

TABLE_DIR.mkdir(parents=True, exist_ok=True)
ANALYSIS_FRAME_DIR.mkdir(parents=True, exist_ok=True)
FINAL_DATA_DIR.mkdir(parents=True, exist_ok=True)

display(Markdown(f"""
## 0. 경로와 로딩 계약

- 원본 파일: `{SOURCE_XLSX}`
- 데이터 시트: `{DATA_SHEET}`
- 실제 컬럼명 행: Excel {HEADER_ROW_EXCEL}행
- 실제 데이터 시작 행: Excel {DATA_START_ROW_EXCEL}행
- 산출 테이블 폴더: `{TABLE_DIR}`
- 분석 프레임 폴더: `{ANALYSIS_FRAME_DIR}`
- 최종 데이터 폴더: `{FINAL_DATA_DIR}`
- 산출 CSV 인코딩: `{OUTPUT_ENCODING}`
"""))



## 0. 경로와 로딩 계약

- 원본 파일: `/home/sieg/projects-wsl/SBS_dataScience/workbook/p2/p2_2/final/data/2024년 고등 학교별X학과별 입학정원 지원 입학 학생 외국인학생 졸업 교원_240912H.xlsx`
- 데이터 시트: `학교별 학과별 주요 현황`
- 실제 컬럼명 행: Excel 14행
- 실제 데이터 시작 행: Excel 15행
- 산출 테이블 폴더: `/home/sieg/projects-wsl/SBS_dataScience/workbook/p2/p2_2/final/eda/P2_G0/tables`
- 분석 프레임 폴더: `/home/sieg/projects-wsl/SBS_dataScience/workbook/p2/p2_2/final/eda/P2_G0/analysis_frames`
- 최종 데이터 폴더: `/home/sieg/projects-wsl/SBS_dataScience/workbook/p2/p2_2/final/eda/P2_G0/final_data`
- 산출 CSV 인코딩: `utf-8-sig`


In [2]:

wb = load_workbook(SOURCE_XLSX, read_only=True, data_only=True)
workbook_profile = pd.DataFrame(
    [
        {
            "sheet_name": ws.title,
            "max_row": ws.max_row,
            "max_column": ws.max_column,
            "selected_for_df": ws.title == DATA_SHEET,
        }
        for ws in wb.worksheets
    ]
)
ws = wb[DATA_SHEET]
header_preview_rows = []
for row_no in range(10, 16):
    values = next(ws.iter_rows(min_row=row_no, max_row=row_no, values_only=True))
    non_null = [
        {"excel_row": row_no, "excel_col": col_no, "value": value}
        for col_no, value in enumerate(values, start=1)
        if value not in (None, "")
    ]
    header_preview_rows.extend(non_null)
wb.close()

header_preview = pd.DataFrame(header_preview_rows)

workbook_profile.to_csv(TABLE_DIR / "00_workbook_profile.csv", index=False, encoding=OUTPUT_ENCODING)
header_preview.to_csv(TABLE_DIR / "01_header_preview_rows_10_to_15.csv", index=False, encoding=OUTPUT_ENCODING)

display(Markdown("## 1. Excel 시트와 헤더 위치 확인"))
display(workbook_profile)
display(Markdown("### 10~15행의 비어 있지 않은 셀"))
display(header_preview.head(180))


## 1. Excel 시트와 헤더 위치 확인

,sheet_name,max_row,max_column,selected_for_df
0,학교별 학과별 주요 현황,35445,129,True
1,용어 정의,17,3,False
2,요약정보,27,13,False


### 10~15행의 비어 있지 않은 셀

,excel_row,excel_col,value
0,10,1,학교 기본 정보
1,10,16,학과정보
2,10,22,학과수 (개)
3,10,25,입학 (명)
4,10,70,재적학생수 (명)
...,...,...,...
175,14,105,외국인유학생_학사_여
176,14,106,외국인유학생_석사_계
177,14,107,외국인유학생_석사_남
178,14,108,외국인유학생_석사_여


In [3]:

def clean_column_name(value: object) -> str:
    text = str(value).replace("\ufeff", "").replace("\n", "_")
    text = re.sub(r"\s+", "", text)
    text = re.sub(r"_+", "_", text)
    return text.strip("_")


def clean_text_series(series: pd.Series) -> pd.Series:
    return (
        series.astype("string")
        .str.replace("\ufeff", "", regex=False)
        .str.replace("\n", " ", regex=False)
        .str.strip()
        .replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})
    )


raw_df = pd.read_excel(
    SOURCE_XLSX,
    sheet_name=DATA_SHEET,
    header=HEADER_ROW_EXCEL - 1,
    engine="openpyxl",
    dtype=object,
)
raw_df.columns = [clean_column_name(col) for col in raw_df.columns]

duplicate_columns = pd.Index(raw_df.columns)[pd.Index(raw_df.columns).duplicated()].tolist()
if duplicate_columns:
    raise ValueError(f"중복 컬럼명이 있습니다: {duplicate_columns[:20]}")

df = raw_df.dropna(how="all").copy()
df = df.loc[:, ~pd.Index(df.columns).astype(str).str.startswith("Unnamed:")]
source_column_count = df.shape[1]
dropped_existing_columns = [col for col in DROPPED_ANALYSIS_COLUMNS if col in df.columns]
dropped_analysis_rows = []
for col in DROPPED_ANALYSIS_COLUMNS:
    exists = col in df.columns
    numeric_values = pd.to_numeric(df[col], errors="coerce") if exists else pd.Series(dtype="float64")
    dropped_analysis_rows.append(
        {
            "column": col,
            "exists_in_source": exists,
            "raw_non_null_n": int(df[col].notna().sum()) if exists else 0,
            "zero_n": int(numeric_values.fillna(0).eq(0).sum()) if exists else 0,
            "nonzero_n": int(numeric_values.fillna(0).ne(0).sum()) if exists else 0,
            "drop_scope": "analysis_df_only",
            "drop_reason": "전 행 0인 시간강사 컬럼이므로 본 EDA 분석 변수에서 제거",
        }
    )
dropped_analysis_columns = pd.DataFrame(dropped_analysis_rows)
df = df.drop(columns=dropped_existing_columns)

TEXT_COLS = [
    "학제",
    "대학원구분",
    "학교명",
    "학교상태",
    "본분교",
    "시도",
    "시군구",
    "설립",
    "주야구분",
    "주소",
    "우편번호",
    "전화번호",
    "팩스번호",
    "홈페이지",
    "학위과정",
    "대계열",
    "중계열",
    "소계열",
    "학과코드",
    "학과명",
]
TEXT_COLS = [col for col in TEXT_COLS if col in df.columns]

for col in TEXT_COLS:
    df[col] = clean_text_series(df[col])

if "연도" in df.columns:
    df["연도"] = pd.to_numeric(df["연도"], errors="coerce").astype("Int64")

NUMERIC_COLS = [col for col in df.columns if col not in [*TEXT_COLS, "연도"]]
conversion_rows = []
for col in NUMERIC_COLS:
    before_non_null = df[col].notna()
    converted = pd.to_numeric(df[col], errors="coerce")
    conversion_rows.append(
        {
            "column": col,
            "raw_non_null": int(before_non_null.sum()),
            "converted_non_null": int(converted.notna().sum()),
            "coerced_to_na": int((before_non_null & converted.isna()).sum()),
        }
    )
    df[col] = converted

conversion_audit = pd.DataFrame(conversion_rows)
problem_conversions = conversion_audit[conversion_audit["coerced_to_na"] > 0].copy()

required_cols = ["연도", "학교명", "본분교", "시도", "학위과정", "대계열", "중계열", "소계열", "학과코드", "학과명"]
missing_required = [col for col in required_cols if col not in df.columns]
if missing_required:
    raise KeyError(f"필수 컬럼 누락: {missing_required}")

overview = pd.DataFrame(
    [
        {"metric": "rows", "value": len(df), "note": "header row excluded; all-empty rows dropped"},
        {"metric": "source_columns", "value": source_column_count, "note": "cleaned source columns before analysis-only drops"},
        {"metric": "dropped_analysis_columns", "value": len(dropped_existing_columns), "note": ", ".join(dropped_existing_columns)},
        {"metric": "columns", "value": df.shape[1], "note": "analysis columns after dropping time-lecturer columns"},
        {"metric": "numeric_columns", "value": len(NUMERIC_COLS), "note": "count/stat fields converted with pd.to_numeric"},
        {"metric": "text_columns", "value": len(TEXT_COLS), "note": "identity/category fields cleaned as pandas string"},
        {"metric": "schools", "value": df["학교명"].nunique(dropna=True), "note": "unique 학교명"},
        {"metric": "departments_raw", "value": df["학과명"].nunique(dropna=True), "note": "unique 학과명"},
        {"metric": "department_codes", "value": df["학과코드"].nunique(dropna=True), "note": "unique 학과코드"},
        {"metric": "provinces", "value": df["시도"].nunique(dropna=True), "note": "unique 시도"},
        {"metric": "major_large_groups", "value": df["대계열"].nunique(dropna=True), "note": "unique 대계열"},
        {"metric": "numeric_conversion_problem_columns", "value": len(problem_conversions), "note": "non-null source values coerced to NA"},
    ]
)

overview.to_csv(TABLE_DIR / "02_dataframe_overview.csv", index=False, encoding=OUTPUT_ENCODING)
conversion_audit.to_csv(TABLE_DIR / "03_numeric_conversion_audit.csv", index=False, encoding=OUTPUT_ENCODING)
df.head(30).to_csv(TABLE_DIR / "04_dataframe_head30.csv", index=False, encoding=OUTPUT_ENCODING)
dropped_analysis_columns.to_csv(TABLE_DIR / "22_dropped_analysis_columns.csv", index=False, encoding=OUTPUT_ENCODING)

display(Markdown("## 2. DataFrame 확정"))
display(overview)
display(Markdown("### df.head(8)"))
display(df.head(8))
display(Markdown("### df.tail(5)"))
display(df.tail(5))
display(Markdown("### 숫자 변환 실패 컬럼"))
display(problem_conversions if len(problem_conversions) else pd.DataFrame([{"status": "숫자 변환 실패 없음"}]))
display(Markdown("### 분석용 DataFrame에서 제거한 컬럼"))
display(dropped_analysis_columns)


## 2. DataFrame 확정

,metric,value,note
0,rows,35431,header row excluded; all-empty rows dropped
1,source_columns,129,cleaned source columns before analysis-only drops
2,dropped_analysis_columns,3,"시간강사_계, 시간강사_남, 시간강사_여"
3,columns,126,analysis columns after dropping time-lecturer ...
4,numeric_columns,105,count/stat fields converted with pd.to_numeric
5,text_columns,20,identity/category fields cleaned as pandas string
6,schools,1693,unique 학교명
7,departments_raw,15044,unique 학과명
8,department_codes,16964,unique 학과코드
9,provinces,17,unique 시도


### df.head(8)

,연도,학제,대학원구분,학교명,학교상태,본분교,시도,시군구,설립,주야구분,주소,우편번호,전화번호,팩스번호,홈페이지,학위과정,대계열,중계열,소계열,학과코드,학과명,학과수_전체,학과수_석사,학과수_박사,입학정원_학부_계,정원내_입학정원_학부,정원외_입학정원_학부,모집인원_학부_계,정원내_모집인원_학부,정원외_모집정원_학부,지원자_전체_계,지원자_전체_남,지원자_전체_여,지원자_석사_계,지원자_석사_남,지원자_석사_여,지원자_박사_계,지원자_박사_남,지원자_박사_여,입학자_전체_계,입학자_전체_남,입학자_전체_여,입학자_석사_계,입학자_석사_남,입학자_석사_여,입학자_박사_계,입학자_박사_남,입학자_박사_여,정원내_입학자_전체_계,정원내_입학자_전체_남,정원내_입학자_전체_여,정원내_입학자_학부_계,정원내_입학자_학부_남,정원내_입학자_학부_여,정원내_입학자_석사_계,정원내_입학자_석사_남,정원내_입학자_석사_여,정원내_입학자_박사_계,정원내_입학자_박사_남,정원내_입학자_박사_여,정원외_입학자_전체_계,정원외_입학자_전체_남,정원외_입학자_전체_여,정원외_입학자_석사_계,정원외_입학자_석사_남,정원외_입학자_석사_여,정원외_입학자_박사_계,정원외_입학자_박사_남,정원외_입학자_박사_여,재적생_전체_계,재적생_전체_남,재적생_전체_여,재학생_전체_계,재학생_전체_남,재학생_전체_여,휴학생_전체_계,휴학생_전체_남,휴학생_전체_여,유예생_전체_계,유예생_전체_남,유예생_전체_여,재적생_석사_계,재적생_석사_남,재적생_석사_여,재학생_석사_계,재학생_석사_남,재학생_석사_여,휴학생_석사_계,휴학생_석사_남,휴학생_석사_여,재적생_박사_계,재적생_박사_남,재적생_박사_여,재학생_박사_계,재학생_박사_남,재학생_박사_여,휴학생_박사_계,휴학생_박사_남,휴학생_박사_여,외국인유학생_총계_계,외국인유학생_총계_남,외국인유학생_총계_여,외국인유학생_학사_계,외국인유학생_학사_남,외국인유학생_학사_여,외국인유학생_석사_계,외국인유학생_석사_남,외국인유학생_석사_여,외국인유학생_박사_계,외국인유학생_박사_남,외국인유학생_박사_여,졸업자_전체,졸업자_남,졸업자_전체_여,졸업자_석사_계,졸업자_석사_남,졸업자_석사_여,졸업자_박사_계,졸업자_박사_남,졸업자_박사_여,전임교원_계,전임교원_남,전임교원_여,비전임교원_계,비전임교원_남,비전임교원_여
0,2024,대학교,<NA>,국립강릉원주대학교,학교명변경,본교(제1캠퍼스),강원,강원 강릉시,국립,주간,"강원도 강릉시 죽헌길 7 (지변동, 강릉원주대학교)",25457,033-640-7001,033-643-7110,www.gwnu.ac.kr,대학과정,인문계열,언어ㆍ문학,국어ㆍ국문학,U01010200008,국어국문학과,1,0,0,29,29,0,29,29,0,111,47,64,0,0,0,0,0,0,31,13,18,0,0,0,0,0,0,29,13,16,29,13,16,0,0,0,0,0,0,2,0,2,0,0,0,0,0,0,152,62,90,113,36,77,39,26,13,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,7,1,6,7,1,6,0,0,0,0,0,0,25,11,14,0,0,0,0,0,0,5,2,3,15,10,5
1,2024,대학교,<NA>,국립강릉원주대학교,학교명변경,본교(제1캠퍼스),강원,강원 강릉시,국립,주간,"강원도 강릉시 죽헌길 7 (지변동, 강릉원주대학교)",25457,033-640-7001,033-643-7110,www.gwnu.ac.kr,대학과정,인문계열,언어ㆍ문학,중국어ㆍ문학,U01010400016,중어중문학과,1,0,0,26,26,0,26,26,0,120,50,70,0,0,0,0,0,0,26,12,14,0,0,0,0,0,0,26,12,14,26,12,14,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,109,47,62,80,35,45,26,11,15,3,1,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,18,6,12,0,0,0,0,0,0,5,4,1,1,0,1
2,2024,대학교,<NA>,국립강릉원주대학교,학교명변경,본교(제1캠퍼스),강원,강원 강릉시,국립,주간,"강원도 강릉시 죽헌길 7 (지변동, 강릉원주대학교)",25457,033-640-7001,033-643-7110,www.gwnu.ac.kr,대학과정,인문계열,언어ㆍ문학,영미어ㆍ문학,U01010600018,영어영문학과,1,0,0,32,32,0,32,32,0,116,54,62,0,0,0,0,0,0,34,17,17,0,0,0,0,0,0,32,15,17,32,15,17,0,0,0,0,0,0,2,2,0,0,0,0,0,0,0,162,76,86,112,44,68,48,32,16,2,0,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,6,0,6,6,0,6,0,0,0,0,0,0,21,8,13,0,0,0,0,0,0,5,3,2,6,5,1
3,2024,대학교,<NA>,국립강릉원주대학교,학교명변경,본교(제1캠퍼스),강원,강원 강릉시,국립,주간,"강원도 강릉시 죽헌길 7 (지변동, 강릉원주대학교)",25457,033-640-7001,033-643-7110,www.gwnu.ac.kr,대학과정,인문계열,언어ㆍ문학,독일어ㆍ문학,U01010700003,독어독문학과,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,39,26,13,19,10,9,17,15,2,3,1,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,10,6,4,0,0,0,0,0,0,1,1,0,5,4,1
4,2024,대학교,<NA>,국립강릉원주대학교,학교명변경,본교(제1캠퍼스),강원,강원 강릉시,국립,주간,"강원도 강릉시 죽헌길 7 (지변동, 강릉원주대학교)",25457,033-640-7001,033-643-7110,www.gwnu.ac.kr,대학과정,인문계열,인문과학,역사ㆍ고고학,U01020400006,사학과,1,0,0,21,21,0,21,21,0,102,64,38,0,0,0,0,0,0,23,15,8,0,0,0,0,0,0,21,13,8,21,13,8,0,0,0,0,0,0,2,2,0,0,0,0,0,0,0,115,91,24,80,56,24,35,35,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,17,10,7,0,0,0,0,0,0,4,4,0,7,6,1
5,2024,대학교,<NA>,국립강릉원주대학교,학교명변경,본교(제1캠퍼스),강원,강원 강릉시,국립,주간,"강원도 강릉시 죽헌길 7 (지변동, 강릉원주대학교)",25457,033-640-7001,033-643-7110,www.gwnu.ac.kr,대학과정,인문계열,인문과학,국제지역학,U01020600060,일본학과,1,0,0,26,26,0,26,26,0,160,86,74,0,0,0,0,0,0,30,14,16,0,0,0,0,0,0,26,12,14,26,12,14,0,0,0,0,0,0,4,2,2,0,0,0,0,0,0,149,86,63,106,58,48,42,27,15,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,13,10,3,0,0,0,0,0,0,5,4,1,2,0,2
6,2024,대학교,<NA>,국립강릉원주대학교,학교명변경,본교(제1캠퍼스),강원,강원 강릉시,국립,주간,"강원도 강릉시 죽헌길 7 (지변동, 강릉원주대학교)",25457,033-640-7001,033-643-7110,www.gwnu.ac.kr,대학과정,인문계열,인문과학,철학ㆍ윤리학,U01020700029,철학과,1,0,0,22,22,0,22,22,0,105,72,33,0,0,0,0,0,0,23,12,11,0,0,0,0,0,0,22,11,11,22,11,11,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,99,65,34,68,41,27,30,23,7,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,

### df.tail(5)

,연도,학제,대학원구분,학교명,학교상태,본분교,시도,시군구,설립,주야구분,주소,우편번호,전화번호,팩스번호,홈페이지,학위과정,대계열,중계열,소계열,학과코드,학과명,학과수_전체,학과수_석사,학과수_박사,입학정원_학부_계,정원내_입학정원_학부,정원외_입학정원_학부,모집인원_학부_계,정원내_모집인원_학부,정원외_모집정원_학부,지원자_전체_계,지원자_전체_남,지원자_전체_여,지원자_석사_계,지원자_석사_남,지원자_석사_여,지원자_박사_계,지원자_박사_남,지원자_박사_여,입학자_전체_계,입학자_전체_남,입학자_전체_여,입학자_석사_계,입학자_석사_남,입학자_석사_여,입학자_박사_계,입학자_박사_남,입학자_박사_여,정원내_입학자_전체_계,정원내_입학자_전체_남,정원내_입학자_전체_여,정원내_입학자_학부_계,정원내_입학자_학부_남,정원내_입학자_학부_여,정원내_입학자_석사_계,정원내_입학자_석사_남,정원내_입학자_석사_여,정원내_입학자_박사_계,정원내_입학자_박사_남,정원내_입학자_박사_여,정원외_입학자_전체_계,정원외_입학자_전체_남,정원외_입학자_전체_여,정원외_입학자_석사_계,정원외_입학자_석사_남,정원외_입학자_석사_여,정원외_입학자_박사_계,정원외_입학자_박사_남,정원외_입학자_박사_여,재적생_전체_계,재적생_전체_남,재적생_전체_여,재학생_전체_계,재학생_전체_남,재학생_전체_여,휴학생_전체_계,휴학생_전체_남,휴학생_전체_여,유예생_전체_계,유예생_전체_남,유예생_전체_여,재적생_석사_계,재적생_석사_남,재적생_석사_여,재학생_석사_계,재학생_석사_남,재학생_석사_여,휴학생_석사_계,휴학생_석사_남,휴학생_석사_여,재적생_박사_계,재적생_박사_남,재적생_박사_여,재학생_박사_계,재학생_박사_남,재학생_박사_여,휴학생_박사_계,휴학생_박사_남,휴학생_박사_여,외국인유학생_총계_계,외국인유학생_총계_남,외국인유학생_총계_여,외국인유학생_학사_계,외국인유학생_학사_남,외국인유학생_학사_여,외국인유학생_석사_계,외국인유학생_석사_남,외국인유학생_석사_여,외국인유학생_박사_계,외국인유학생_박사_남,외국인유학생_박사_여,졸업자_전체,졸업자_남,졸업자_전체_여,졸업자_석사_계,졸업자_석사_남,졸업자_석사_여,졸업자_박사_계,졸업자_박사_남,졸업자_박사_여,전임교원_계,전임교원_남,전임교원_여,비전임교원_계,비전임교원_남,비전임교원_여
35426,2024,기능대학,<NA>,한국폴리텍대학 항공캠퍼스,기존,본교(제1캠퍼스),경남,경남 사천시,사립,주간,"경상남도 사천시 대학길 46 (이금동, 한국폴리텍항공대학)",52549,055-830-3400,055-830-3414,www.kopo.ac.kr/kapc,전문대학과정,공학계열,교통ㆍ운송,항공,C04030200097,항공모빌리티과(스마트기계시스템),1,0,0,0,0,0,15,0,15,20,20,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,15,15,0,15,15,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
35427,2024,기능대학,<NA>,한국폴리텍대학 항공캠퍼스,기존,본교(제1캠퍼스),경남,경남 사천시,사립,주간,"경상남도 사천시 대학길 46 (이금동, 한국폴리텍항공대학)",52549,055-830-3400,055-830-3414,www.kopo.ac.kr/kapc,전문대학과정,공학계열,교통ㆍ운송,항공,C04030200099,항공모빌리티과,1,0,0,25,25,0,30,25,5,56,49,7,0,0,0,0,0,0,22,20,2,0,0,0,0,0,0,21,19,2,21,19,2,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,20,18,2,20,18,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,3,3,0,0,0,0
35428,2024,기능대학,<NA>,한국폴리텍대학 항공캠퍼스,기존,본교(제1캠퍼스),경남,경남 사천시,사립,주간,"경상남도 사천시 대학길 46 (이금동, 한국폴리텍항공대학)",52549,055-830-3400,055-830-3414,www.kopo.ac.kr/kapc,전문대학과정,공학계열,교통ㆍ운송,항공,C04030200102,항공모빌리티정비과,1,0,0,25,25,0,30,25,5,39,36,3,0,0,0,0,0,0,20,20,0,0,0,0,0,0,0,17,17,0,17,17,0,0,0,0,0,0,0,3,3,0,0,0,0,0,0,0,20,20,0,20,20,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2,2,0,5,5,0
35429,2024,기능대학,<NA>,한국폴리텍대학 항공캠퍼스,기존,본교(제1캠퍼스),경남,경남 사천시,사립,주간,"경상남도 사천시 대학길 46 (이금동, 한국폴리텍항공대학)",52549,055-830-3400,055-830-3414,www.kopo.ac.kr/kapc,전문대학과정,공학계열,전기ㆍ전자,제어계측,C04050300019,항공전기제어과,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
35430,2024,기능대학,<NA>,한국폴리텍대학 항공캠퍼스,기존,본교(제1캠퍼스),경남,경남 사천시,사립,주간,"경상남도 사천시 대학길 46 (이금동, 한국폴리텍항공대학)",52549,055-830-3400,055-830-3414,www.kopo.ac.kr/kapc,전문대학과정,<NA>,<NA>,<NA>,Z99999999999,소속학과없음,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,1,1,0


### 숫자 변환 실패 컬럼

,status
0,숫자 변환 실패 없음


### 분석용 DataFrame에서 제거한 컬럼

,column,exists_in_source,raw_non_null_n,zero_n,nonzero_n,drop_scope,drop_reason
0,시간강사_계,True,35431,35431,0,analysis_df_only,전 행 0인 시간강사 컬럼이므로 본 EDA 분석 변수에서 제거
1,시간강사_남,True,35431,35431,0,analysis_df_only,전 행 0인 시간강사 컬럼이므로 본 EDA 분석 변수에서 제거
2,시간강사_여,True,35431,35431,0,analysis_df_only,전 행 0인 시간강사 컬럼이므로 본 EDA 분석 변수에서 제거


In [4]:

def sample_values(series: pd.Series, n: int = 5) -> str:
    values = series.dropna().astype(str).drop_duplicates().head(n).tolist()
    return " | ".join(values)


role_lookup = {}
for col in TEXT_COLS:
    role_lookup[col] = "id/category"
if "연도" in df.columns:
    role_lookup["연도"] = "id/time"
for col in NUMERIC_COLS:
    role_lookup[col] = "measure"

column_profile = pd.DataFrame(
    [
        {
            "column_order": idx,
            "column": col,
            "role": role_lookup.get(col, "unknown"),
            "dtype": str(df[col].dtype),
            "non_null_n": int(df[col].notna().sum()),
            "missing_n": int(df[col].isna().sum()),
            "missing_pct": float(df[col].isna().mean() * 100),
            "unique_n": int(df[col].nunique(dropna=True)),
            "sample_values": sample_values(df[col]),
        }
        for idx, col in enumerate(df.columns, start=1)
    ]
)
column_profile.to_csv(TABLE_DIR / "05_column_profile.csv", index=False, encoding=OUTPUT_ENCODING)

display(Markdown("## 3. 컬럼 프로파일"))
display(column_profile)
display(Markdown("### 결측률 상위 25개 컬럼"))
display(column_profile.sort_values(["missing_pct", "missing_n"], ascending=False).head(25))


## 3. 컬럼 프로파일

,column_order,column,role,dtype,non_null_n,missing_n,missing_pct,unique_n,sample_values
0,1,연도,id/time,Int64,35431,0,0.000,1,2024
1,2,학제,id/category,string,35431,0,0.000,18,대학교 | 일반대학원 | 특수대학원 | 전문대학원 | 교육대학
2,3,대학원구분,id/category,string,12674,22757,64.229,2,부설대학원 | 대학원대학
3,4,학교명,id/category,string,35431,0,0.000,1693,국립강릉원주대학교 | 국립강릉원주대학교대학원 | 국립강릉원주대학교경영정책과학대학원 ...
4,5,학교상태,id/category,string,35431,0,0.000,4,학교명변경 | 기존 | 폐교 | 신설
...,...,...,...,...,...,...,...,...,...
121,122,전임교원_남,measure,int64,35431,0,0.000,100,2 | 4 | 3 | 1 | 0
122,123,전임교원_여,measure,int64,35431,0,0.000,69,3 | 1 | 2 | 0 | 4
123,124,비전임교원_계,measure,int64,35431,0,0.000,154,15 | 1 | 6 | 5 | 7
124,125,비전임교원_남,measure,int64,35431,0,0.000,112,10 | 0 | 5 | 4 | 6


### 결측률 상위 25개 컬럼

,column_order,column,role,dtype,non_null_n,missing_n,missing_pct,unique_n,sample_values
2,3,대학원구분,id/category,string,12674,22757,64.229,2,부설대학원 | 대학원대학
13,14,팩스번호,id/category,string,30613,4818,13.598,900,033-643-7110 | 033-760-8019 | 033-251-9556 | 0...
12,13,전화번호,id/category,string,34189,1242,3.505,1065,033-640-7001 | 033-760-8114 | 033-640-2077 | 0...
14,15,홈페이지,id/category,string,34418,1013,2.859,1233,www.gwnu.ac.kr | www.kangwon.ac.kr/ | graduate...
16,17,대계열,id/category,string,34969,462,1.304,7,인문계열 | 사회계열 | 교육계열 | 공학계열 | 자연계열
17,18,중계열,id/category,string,34969,462,1.304,36,언어ㆍ문학 | 인문과학 | 경영ㆍ경제 | 법률 | 사회과학
18,19,소계열,id/category,string,34969,462,1.304,188,국어ㆍ국문학 | 중국어ㆍ문학 | 영미어ㆍ문학 | 독일어ㆍ문학 | 역사ㆍ고고학
0,1,연도,id/time,Int64,35431,0,0.000,1,2024
1,2,학제,id/category,string,35431,0,0.000,18,대학교 | 일반대학원 | 특수대학원 | 전문대학원 | 교육대학
3,4,학교명,id/category,string,35431,0,0.000,1693,국립강릉원주대학교 | 국립강릉원주대학교대학원 | 국립강릉원주대학교경영정책과학대학원 ...


In [5]:

numeric_stats = df[NUMERIC_COLS].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).T
numeric_stats["missing_n"] = df[NUMERIC_COLS].isna().sum()
numeric_stats["missing_pct"] = numeric_stats["missing_n"] / len(df) * 100
numeric_stats["zero_n"] = (df[NUMERIC_COLS] == 0).sum()
numeric_stats["negative_n"] = (df[NUMERIC_COLS] < 0).sum()
numeric_stats["skew"] = df[NUMERIC_COLS].skew(numeric_only=True)
numeric_stats["kurtosis"] = df[NUMERIC_COLS].kurt(numeric_only=True)
numeric_stats = numeric_stats.reset_index().rename(columns={"index": "column"})

display_cols = ["column", "count", "mean", "std", "min", "1%", "5%", "25%", "50%", "75%", "95%", "99%", "max", "missing_n", "missing_pct", "zero_n", "negative_n", "skew"]
numeric_stats_view = numeric_stats[display_cols].copy()
round_cols = [col for col in numeric_stats_view.columns if col not in ["column", "count", "missing_n", "zero_n", "negative_n"]]
for col in round_cols:
    numeric_stats_view[col] = pd.to_numeric(numeric_stats_view[col], errors="coerce").round(3)

numeric_stats.to_csv(TABLE_DIR / "06_numeric_descriptive_stats.csv", index=False, encoding=OUTPUT_ENCODING)
numeric_stats_view.to_csv(TABLE_DIR / "07_numeric_descriptive_stats_view.csv", index=False, encoding=OUTPUT_ENCODING)

display(Markdown("## 4. 수치형 기술통계"))
display(numeric_stats_view)
display(Markdown("### 0 값이 많은 수치 컬럼 상위 30개"))
display(numeric_stats_view.sort_values("zero_n", ascending=False).head(30))


## 4. 수치형 기술통계

,column,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max,missing_n,missing_pct,zero_n,negative_n,skew
0,학과수_전체,"35,431.000",1.031,0.606,0.000,0.000,0.000,1.000,1.000,1.000,2.000,2.000,8.000,0,0.000,5590,0,0.385
1,학과수_석사,"35,431.000",0.296,0.467,0.000,0.000,0.000,0.000,0.000,1.000,1.000,1.000,3.000,0,0.000,25104,0,1.034
2,학과수_박사,"35,431.000",0.151,0.359,0.000,0.000,0.000,0.000,0.000,0.000,1.000,1.000,3.000,0,0.000,30074,0,1.965
3,입학정원_학부_계,"35,431.000",16.116,85.263,0.000,0.000,0.000,0.000,0.000,18.000,70.000,151.000,"4,500.000",0,0.000,25694,0,30.549
4,정원내_입학정원_학부,"35,431.000",15.522,85.086,0.000,0.000,0.000,0.000,0.000,10.000,70.000,150.000,"4,500.000",0,0.000,26349,0,30.752
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
100,전임교원_남,"35,431.000",1.728,8.790,0.000,0.000,0.000,0.000,0.000,1.000,7.000,20.000,622.000,0,0.000,24446,0,34.194
101,전임교원_여,"35,431.000",0.737,3.804,0.000,0.000,0.000,0.000,0.000,0.000,4.000,10.000,281.000,0,0.000,27489,0,30.584
102,비전임교원_계,"35,431.000",4.224,13.658,0.000,0.000,0.000,0.000,0.000,4.000,18.000,45.000,854.000,0,0.000,19677,0,23.388
103,비전임교원_남,"35,431.000",2.402,8.644,0.000,0.000,0.000,0.000,0.000,2.000,11.000,26.000,565.000,0,0.000,21745,0,27.598


### 0 값이 많은 수치 컬럼 상위 30개

,column,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max,missing_n,missing_pct,zero_n,negative_n,skew
5,정원외_입학정원_학부,"35,431.000",0.593,4.270,0.000,0.000,0.000,0.000,0.000,0.000,0.000,20.000,209.000,0,0.000,34481,0,12.586
47,정원외_입학자_박사_여,"35,431.000",0.113,1.217,0.000,0.000,0.000,0.000,0.000,0.000,0.000,3.000,93.000,0,0.000,34180,0,33.422
46,정원외_입학자_박사_남,"35,431.000",0.112,1.182,0.000,0.000,0.000,0.000,0.000,0.000,0.000,3.000,83.000,0,0.000,34144,0,32.422
88,외국인유학생_박사_남,"35,431.000",0.244,2.488,0.000,0.000,0.000,0.000,0.000,0.000,1.000,6.000,184.000,0,0.000,33589,0,32.011
45,정원외_입학자_박사_계,"35,431.000",0.225,2.276,0.000,0.000,0.000,0.000,0.000,0.000,1.000,5.000,158.000,0,0.000,33584,0,33.965
77,휴학생_박사_여,"35,431.000",0.136,1.072,0.000,0.000,0.000,0.000,0.000,0.000,1.000,3.000,81.000,0,0.000,33568,0,28.968
43,정원외_입학자_석사_남,"35,431.000",0.295,3.314,0.000,0.000,0.000,0.000,0.000,0.000,1.000,6.000,229.000,0,0.000,33544,0,32.515
89,외국인유학생_박사_여,"35,431.000",0.272,3.067,0.000,0.000,0.000,0.000,0.000,0.000,1.000,6.000,258.000,0,0.000,33510,0,39.887
76,휴학생_박사_남,"35,431.000",0.148,1.200,0.000,0.000,0.000,0.000,0.000,0.000,1.000,3.000,95.000,0,0.000,33389,0,33.486
58,유예생_전체_남,"35,431.000",0.192,1.437,0.000,0.000,0.000,0.000,0.000,0.000,1.000,5.000,67.000,0,0.000,33295,0,19.336


In [6]:

CATEGORICAL_AUDIT_COLS = [
    "연도",
    "학제",
    "대학원구분",
    "학교상태",
    "본분교",
    "시도",
    "설립",
    "주야구분",
    "학위과정",
    "대계열",
    "중계열",
    "소계열",
]
CATEGORICAL_AUDIT_COLS = [col for col in CATEGORICAL_AUDIT_COLS if col in df.columns]

categorical_rows = []
for col in CATEGORICAL_AUDIT_COLS:
    counts = df[col].astype("string").fillna("결측").value_counts(dropna=False).head(30)
    for rank, (value, count) in enumerate(counts.items(), start=1):
        categorical_rows.append(
            {
                "column": col,
                "rank": rank,
                "value": value,
                "row_count": int(count),
                "row_pct": float(count / len(df) * 100),
            }
        )

categorical_top_counts = pd.DataFrame(categorical_rows)
categorical_top_counts.to_csv(TABLE_DIR / "08_categorical_top_counts.csv", index=False, encoding=OUTPUT_ENCODING)

display(Markdown("## 5. 주요 범주형 분포"))
display(categorical_top_counts)


## 5. 주요 범주형 분포

,column,rank,value,row_count,row_pct
0,연도,1,2024,35431,100.000
1,학제,1,대학교,14823,41.836
2,학제,2,일반대학원,6980,19.700
3,학제,3,전문대학,6545,18.473
4,학제,4,특수대학원,4827,13.624
...,...,...,...,...,...
132,소계열,26,재활학,369,1.041
133,소계열,27,국어ㆍ국문학,364,1.027
134,소계열,28,토목공학,348,0.982
135,소계열,29,법학,344,0.971


In [7]:

candidate_keys = {
    "year_school_campus_degree_deptcode": ["연도", "학교명", "본분교", "학위과정", "학과코드"],
    "year_school_campus_day_degree_deptcode": ["연도", "학교명", "본분교", "주야구분", "학위과정", "학과코드"],
    "year_school_campus_day_degree_deptcode_deptname": ["연도", "학교명", "본분교", "주야구분", "학위과정", "학과코드", "학과명"],
    "year_school_campus_day_degree_series_mid_small_deptname": ["연도", "학교명", "본분교", "주야구분", "학위과정", "대계열", "중계열", "소계열", "학과명"],
}

key_rows = []
duplicate_examples = []
for key_name, cols in candidate_keys.items():
    existing_cols = [col for col in cols if col in df.columns]
    dup_mask = df.duplicated(existing_cols, keep=False)
    key_rows.append(
        {
            "key_name": key_name,
            "columns": " + ".join(existing_cols),
            "missing_key_columns": " + ".join([col for col in cols if col not in df.columns]),
            "duplicate_rows": int(dup_mask.sum()),
            "duplicate_groups": int(df.loc[dup_mask].groupby(existing_cols, dropna=False).ngroups) if dup_mask.any() else 0,
        }
    )
    if dup_mask.any():
        ex = df.loc[dup_mask, existing_cols + ["시도", "학제", "학교상태"]].head(30).copy()
        ex.insert(0, "key_name", key_name)
        duplicate_examples.append(ex)

key_audit = pd.DataFrame(key_rows)
duplicate_key_examples = pd.concat(duplicate_examples, ignore_index=True) if duplicate_examples else pd.DataFrame()

key_audit.to_csv(TABLE_DIR / "09_candidate_key_audit.csv", index=False, encoding=OUTPUT_ENCODING)
duplicate_key_examples.to_csv(TABLE_DIR / "10_duplicate_key_examples.csv", index=False, encoding=OUTPUT_ENCODING)

school_summary = (
    df.groupby(["시도", "학교명"], dropna=False)
    .agg(
        rows=("학과명", "count"),
        dept_codes=("학과코드", "nunique"),
        degree_courses=("학위과정", "nunique"),
        total_capacity=("입학정원_학부_계", "sum"),
        total_applicants=("지원자_전체_계", "sum"),
        total_admitted=("입학자_전체_계", "sum"),
        total_enrolled=("재적생_전체_계", "sum"),
        total_foreign_students=("외국인유학생_총계_계", "sum"),
        total_graduates=("졸업자_전체", "sum"),
        full_time_faculty=("전임교원_계", "sum"),
    )
    .reset_index()
    .sort_values(["rows", "dept_codes"], ascending=False)
)
school_summary.to_csv(TABLE_DIR / "11_school_summary.csv", index=False, encoding=OUTPUT_ENCODING)

display(Markdown("## 6. 후보 키 중복 점검"))
display(key_audit)
display(Markdown("### 중복 키 예시"))
display(duplicate_key_examples if len(duplicate_key_examples) else pd.DataFrame([{"status": "후보 키 중복 예시 없음"}]))
display(Markdown("### 학교별 요약 상위 30개"))
display(school_summary.head(30))


## 6. 후보 키 중복 점검

,key_name,columns,missing_key_columns,duplicate_rows,duplicate_groups
0,year_school_campus_degree_deptcode,연도 + 학교명 + 본분교 + 학위과정 + 학과코드,,0,0
1,year_school_campus_day_degree_deptcode,연도 + 학교명 + 본분교 + 주야구분 + 학위과정 + 학과코드,,0,0
2,year_school_campus_day_degree_deptcode_deptname,연도 + 학교명 + 본분교 + 주야구분 + 학위과정 + 학과코드 + 학과명,,0,0
3,year_school_campus_day_degree_series_mid_small...,연도 + 학교명 + 본분교 + 주야구분 + 학위과정 + 대계열 + 중계열 + 소계열...,,0,0


### 중복 키 예시

,status
0,후보 키 중복 예시 없음


### 학교별 요약 상위 30개

,시도,학교명,rows,dept_codes,degree_courses,total_capacity,total_applicants,total_admitted,total_enrolled,total_foreign_students,total_graduates,full_time_faculty
7,강원,강원대학교,358,337,1,4465,31613,4806,28770,250,4243,881
471,경북,대구대학교,206,206,1,3693,17955,3851,20397,1426,3649,504
849,부산,신라대학교,198,198,1,1640,8888,1540,8383,312,1695,276
371,경남,경상국립대학교,194,192,1,4261,28729,4545,21100,85,2910,1029
465,경북,대구가톨릭대학교,191,191,1,2289,14210,2476,13341,249,2230,319
842,부산,부산외국어대학교,183,183,1,1470,5687,1606,9532,565,1531,218
777,부산,국립부경대학교,181,181,1,3298,25670,3682,20749,671,3461,629
810,부산,동의대학교,177,177,1,3514,19968,3864,19490,206,3241,553
1539,전북,전북대학교,177,177,1,3944,30090,4331,23523,603,4042,1003
632,대구,계명대학교,176,175,1,4619,30946,5017,28685,1118,4707,840


## 7. 0 / NULL / 해당없음 구분 원칙

이 단계에서는 원자료의 빈칸을 모두 같은 결측으로 처리하지 않는다.

- `0`: 수치형 집계에서 실제 0명/0개/0건을 뜻하는 값으로 유지한다.
- `해당없음`: 구조상 값이 있을 수 없는 빈칸이다. 예: 비대학원 행의 `대학원구분`.
- `구조적 placeholder`: `소속학과없음`처럼 학과 행처럼 보이지만 실제 학과 분석 단위가 아닌 행이다. 일부 교원 수치가 배정될 수 있으므로 총합 분석에서는 따로 다룬다.
- `실질결측`: 해당 행에서 값이 있어야 하는데 빠진 값이다.

분석 단위가 학과일 때는 `소속학과없음` 행을 평균/분포 계산에서 제외하는 것이 안전하다.


In [8]:

SIMPLE_CATEGORY_COLS = [
    "학제",
    "학교상태",
    "본분교",
    "시도",
    "시군구",
    "설립",
    "주야구분",
    "학위과정",
]
CONDITIONAL_CATEGORY_COLS = ["대학원구분"]
TAXONOMY_COLS = ["대계열", "중계열", "소계열"]
CONTACT_OPTIONAL_COLS = ["전화번호", "팩스번호", "홈페이지"]
CONTACT_ID_COLS = ["주소", "우편번호", *CONTACT_OPTIONAL_COLS]
ENTITY_ID_COLS = ["학교명", "학과코드", "학과명"]


def measure_semantic_type(column: str) -> str:
    if column.startswith("학과수"):
        return "department_count"
    if any(token in column for token in ["입학정원", "모집인원", "지원자", "입학자"]):
        return "competitive_admission_count"
    if any(token in column for token in ["재적생", "재학생", "휴학생", "유예생"]):
        return "student_stock_count"
    if "외국인유학생" in column:
        return "foreign_student_count"
    if "졸업자" in column:
        return "graduate_outcome_count"
    if any(token in column for token in ["교원", "시간강사"]):
        return "faculty_count"
    return "numeric_measure"


def semantic_type(column: str) -> str:
    if column == "연도":
        return "time_key"
    if column in ENTITY_ID_COLS:
        return "entity_identifier"
    if column in CONTACT_ID_COLS:
        return "contact_reference_optional"
    if column in CONDITIONAL_CATEGORY_COLS:
        return "conditional_category"
    if column in TAXONOMY_COLS:
        return "taxonomy_category"
    if column in SIMPLE_CATEGORY_COLS:
        return "simple_category"
    if column in NUMERIC_COLS:
        return measure_semantic_type(column)
    return "other"


def category_behavior(column: str) -> str:
    stype = semantic_type(column)
    if stype == "competitive_admission_count":
        return "경쟁형 수치: 정원-모집-지원-입학 흐름에서 0도 의미 있는 관측값"
    if stype in {"student_stock_count", "foreign_student_count", "graduate_outcome_count", "faculty_count", "department_count", "numeric_measure"}:
        return "집계형 수치: 0은 없음/미발생으로 유지"
    if stype == "conditional_category":
        return "조건부 범주: 대학원 행에만 값이 필요하고 비대학원 행의 빈칸은 해당없음"
    if stype == "taxonomy_category":
        return "분류 범주: 실제 학과 행에서는 값 필요, 소속학과없음 행은 구조적 placeholder"
    if stype == "simple_category":
        return "단순 범주: 집단 구분용 범주값"
    if stype == "contact_reference_optional":
        return "참조 정보: 분석 핵심 변수는 아니며 빈칸은 연락처/URL 누락"
    if stype == "entity_identifier":
        return "식별자: 키 구성과 중복 점검에 필요"
    return "기타"


def zero_policy(column: str) -> str:
    stype = semantic_type(column)
    if stype == "competitive_admission_count":
        return "0 유지: 미모집/미지원/미입학 가능성이 있으므로 NULL로 바꾸지 않음"
    if stype in {"student_stock_count", "foreign_student_count", "graduate_outcome_count", "faculty_count", "department_count", "numeric_measure"}:
        return "0 유지: 집계값의 실제 0으로 해석"
    return "해당없음"


def null_policy(column: str) -> str:
    stype = semantic_type(column)
    if column == "대학원구분":
        return "학위과정=대학원과정이면 값 필요, 그 외 NULL은 해당없음"
    if column in TAXONOMY_COLS:
        return "실제 학과 행이면 값 필요, 소속학과없음 placeholder의 NULL은 구조적"
    if column in CONTACT_OPTIONAL_COLS:
        return "분석 핵심 결측은 아니고 참조정보 누락으로 분리"
    if stype.startswith("numeric") or stype.endswith("_count"):
        return "NULL이면 데이터 품질 문제, 0은 유효값"
    return "NULL이면 값 누락 여부 확인"


is_graduate_course = df["학위과정"].astype("string").eq("대학원과정").fillna(False)
is_no_department_placeholder = (
    df["학과코드"].astype("string").eq("Z99999999999")
    | df["학과명"].astype("string").eq("소속학과없음")
).fillna(False)
measure_nonzero_n = df[NUMERIC_COLS].fillna(0).ne(0).sum(axis=1)

semantic_contract = pd.DataFrame(
    [
        {
            "column_order": idx,
            "column": col,
            "semantic_type": semantic_type(col),
            "category_behavior": category_behavior(col),
            "zero_policy": zero_policy(col),
            "null_policy": null_policy(col),
        }
        for idx, col in enumerate(df.columns, start=1)
    ]
)
semantic_contract.to_csv(TABLE_DIR / "12_column_semantic_contract.csv", index=False, encoding=OUTPUT_ENCODING)

missing_rows = []
sample_rows = []
for col in df.columns:
    raw_null = df[col].isna()
    structural_null = pd.Series(False, index=df.index)
    optional_null = pd.Series(False, index=df.index)

    if col == "대학원구분":
        structural_null = raw_null & ~is_graduate_course
    elif col in TAXONOMY_COLS:
        structural_null = raw_null & is_no_department_placeholder
    elif col in CONTACT_OPTIONAL_COLS:
        optional_null = raw_null

    true_missing = raw_null & ~structural_null & ~optional_null
    zero_n = int((df[col] == 0).sum()) if col in NUMERIC_COLS else 0
    all_zero = bool(col in NUMERIC_COLS and df[col].notna().all() and (df[col] == 0).all())

    if true_missing.sum() > 0:
        risk_level = "high"
        cleaning_action = "실질결측: 원자료 확인 또는 분석 제외/보정 필요"
    elif optional_null.sum() > 0:
        risk_level = "low"
        cleaning_action = "참조정보 누락: 분석 변수에서는 제외 가능"
    elif structural_null.sum() > 0:
        risk_level = "low"
        cleaning_action = "구조적 해당없음/placeholder로 별도 표기"
    elif all_zero:
        risk_level = "medium"
        cleaning_action = "전 컬럼 0: 원자료 정의 확인 후 0 유지"
    else:
        risk_level = "ok"
        cleaning_action = "그대로 사용"

    missing_rows.append(
        {
            "column": col,
            "semantic_type": semantic_type(col),
            "raw_null_n": int(raw_null.sum()),
            "raw_null_pct": float(raw_null.mean() * 100),
            "structural_null_n": int(structural_null.sum()),
            "optional_reference_null_n": int(optional_null.sum()),
            "true_missing_n": int(true_missing.sum()),
            "true_missing_pct": float(true_missing.mean() * 100),
            "zero_n": zero_n,
            "zero_pct": float((df[col] == 0).mean() * 100) if col in NUMERIC_COLS else np.nan,
            "all_zero_column": all_zero,
            "risk_level": risk_level,
            "cleaning_action": cleaning_action,
        }
    )

    if raw_null.any():
        if col == "대학원구분":
            reason = "비대학원 행이면 해당없음, 대학원 행이면 실질결측"
        elif col in TAXONOMY_COLS:
            reason = "소속학과없음 placeholder이면 구조적, 실제 학과이면 실질결측"
        elif col in CONTACT_OPTIONAL_COLS:
            reason = "연락처/홈페이지 참조정보 누락"
        else:
            reason = "값 필요 여부 검토"
        sample_cols = [
            "연도",
            "학제",
            "대학원구분",
            "학교명",
            "본분교",
            "시도",
            "학위과정",
            "대계열",
            "중계열",
            "소계열",
            "학과코드",
            "학과명",
            "학과수_전체",
            "입학정원_학부_계",
            "지원자_전체_계",
            "재적생_전체_계",
        ]
        sample_cols = [sample_col for sample_col in sample_cols if sample_col in df.columns]
        sample = df.loc[raw_null, sample_cols].head(12).copy()
        sample.insert(0, "null_column", col)
        sample.insert(1, "null_interpretation", reason)
        sample_rows.append(sample)

effective_missingness = pd.DataFrame(missing_rows)
effective_missingness = effective_missingness.sort_values(
    ["true_missing_n", "raw_null_n", "zero_n"],
    ascending=[False, False, False],
).reset_index(drop=True)
effective_missing_samples = pd.concat(sample_rows, ignore_index=True) if sample_rows else pd.DataFrame()

numeric_zero_policy_audit = effective_missingness[
    effective_missingness["semantic_type"].isin(
        [
            "department_count",
            "competitive_admission_count",
            "student_stock_count",
            "foreign_student_count",
            "graduate_outcome_count",
            "faculty_count",
            "numeric_measure",
        ]
    )
].copy()
numeric_zero_policy_audit = numeric_zero_policy_audit[
    ["column", "semantic_type", "zero_n", "zero_pct", "all_zero_column", "zero_n", "risk_level", "cleaning_action"]
].loc[:, lambda frame: ~frame.columns.duplicated()]
numeric_zero_policy_audit = numeric_zero_policy_audit.sort_values(["all_zero_column", "zero_pct"], ascending=[False, False])

placeholder_numeric_sums = df.loc[is_no_department_placeholder, NUMERIC_COLS].fillna(0).sum()
placeholder_nonzero_measure_audit = (
    placeholder_numeric_sums[placeholder_numeric_sums.ne(0)]
    .rename("placeholder_total")
    .reset_index()
    .rename(columns={"index": "column"})
)
if len(placeholder_nonzero_measure_audit):
    placeholder_nonzero_measure_audit["nonzero_rows"] = placeholder_nonzero_measure_audit["column"].map(
        lambda col: int(df.loc[is_no_department_placeholder, col].fillna(0).ne(0).sum())
    )
    placeholder_nonzero_measure_audit["semantic_type"] = placeholder_nonzero_measure_audit["column"].map(semantic_type)
    placeholder_nonzero_measure_audit["handling_note"] = placeholder_nonzero_measure_audit["semantic_type"].map(
        lambda stype: "학과별 분포/평균에서는 제외, 총합에서는 소속학과없음 교원 항목으로 별도 집계"
        if stype == "faculty_count"
        else "학과별 분석 제외 전 별도 검토"
    )
    placeholder_nonzero_measure_audit = placeholder_nonzero_measure_audit.sort_values("placeholder_total", ascending=False)
else:
    placeholder_nonzero_measure_audit = pd.DataFrame(
        columns=["column", "placeholder_total", "nonzero_rows", "semantic_type", "handling_note"]
    )

df_cleaning_flags = df[
    [
        "연도",
        "학제",
        "대학원구분",
        "학교명",
        "본분교",
        "시도",
        "학위과정",
        "대계열",
        "중계열",
        "소계열",
        "학과코드",
        "학과명",
    ]
].copy()
df_cleaning_flags["is_graduate_course"] = is_graduate_course
df_cleaning_flags["is_no_department_placeholder"] = is_no_department_placeholder
df_cleaning_flags["measure_nonzero_n"] = measure_nonzero_n
df_cleaning_flags["대학원구분_clean"] = np.where(
    is_graduate_course,
    df["대학원구분"].astype("string").fillna("실질결측_대학원구분"),
    df["대학원구분"].astype("string").fillna("해당없음_비대학원"),
)
for col in TAXONOMY_COLS:
    df_cleaning_flags[f"{col}_clean"] = df[col].astype("string")
    df_cleaning_flags.loc[is_no_department_placeholder & df[col].isna(), f"{col}_clean"] = "소속학과없음"
    df_cleaning_flags[f"{col}_clean"] = df_cleaning_flags[f"{col}_clean"].fillna("실질결측_계열미상")

row_cleaning_summary = pd.DataFrame(
    [
        {
            "check": "non_graduate_rows_with_null_graduate_type",
            "row_count": int((~is_graduate_course & df["대학원구분"].isna()).sum()),
            "interpretation": "대학원구분 해당없음: NULL을 '해당없음_비대학원'으로 분석용 보정",
        },
        {
            "check": "graduate_rows_with_null_graduate_type",
            "row_count": int((is_graduate_course & df["대학원구분"].isna()).sum()),
            "interpretation": "대학원구분 실질결측",
        },
        {
            "check": "no_department_placeholder_rows",
            "row_count": int(is_no_department_placeholder.sum()),
            "interpretation": "학과명/계열별 평균·분포에서는 제외, 총합 분석에서는 소속학과없음 항목으로 별도 집계",
        },
        {
            "check": "no_department_placeholder_rows_with_nonzero_measures",
            "row_count": int((is_no_department_placeholder & measure_nonzero_n.gt(0)).sum()),
            "interpretation": "placeholder 행의 비0 수치 존재 여부. 현재는 교원 계열 수치가 배정되어 별도 처리 필요",
        },
        {
            "check": "taxonomy_true_missing_rows",
            "row_count": int(((df[TAXONOMY_COLS].isna().any(axis=1)) & ~is_no_department_placeholder).sum()),
            "interpretation": "실제 학과 행의 계열 실질결측",
        },
    ]
)

effective_missingness.to_csv(TABLE_DIR / "13_effective_missingness_by_column.csv", index=False, encoding=OUTPUT_ENCODING)
effective_missing_samples.to_csv(TABLE_DIR / "14_effective_missing_samples.csv", index=False, encoding=OUTPUT_ENCODING)
numeric_zero_policy_audit.to_csv(TABLE_DIR / "15_numeric_zero_policy_audit.csv", index=False, encoding=OUTPUT_ENCODING)
df_cleaning_flags.to_csv(TABLE_DIR / "16_row_cleaning_flags.csv", index=False, encoding=OUTPUT_ENCODING)
row_cleaning_summary.to_csv(TABLE_DIR / "17_row_cleaning_summary.csv", index=False, encoding=OUTPUT_ENCODING)
placeholder_nonzero_measure_audit.to_csv(TABLE_DIR / "21_placeholder_nonzero_measure_audit.csv", index=False, encoding=OUTPUT_ENCODING)

display(Markdown("## 8. 컬럼 의미 계약: 경쟁형/범주형/조건부 범주 구분"))
display(semantic_contract)
display(Markdown("## 9. 실질 결측 산출"))
display(effective_missingness[effective_missingness["raw_null_n"].gt(0) | effective_missingness["all_zero_column"]].head(80))
display(Markdown("### NULL 샘플 행"))
display(effective_missing_samples.head(80))
display(Markdown("### 행 단위 클리닝 요약"))
display(row_cleaning_summary)
display(Markdown("### `소속학과없음` placeholder에 배정된 비0 수치"))
display(placeholder_nonzero_measure_audit)


## 8. 컬럼 의미 계약: 경쟁형/범주형/조건부 범주 구분

,column_order,column,semantic_type,category_behavior,zero_policy,null_policy
0,1,연도,time_key,기타,해당없음,NULL이면 값 누락 여부 확인
1,2,학제,simple_category,단순 범주: 집단 구분용 범주값,해당없음,NULL이면 값 누락 여부 확인
2,3,대학원구분,conditional_category,조건부 범주: 대학원 행에만 값이 필요하고 비대학원 행의 빈칸은 해당없음,해당없음,"학위과정=대학원과정이면 값 필요, 그 외 NULL은 해당없음"
3,4,학교명,entity_identifier,식별자: 키 구성과 중복 점검에 필요,해당없음,NULL이면 값 누락 여부 확인
4,5,학교상태,simple_category,단순 범주: 집단 구분용 범주값,해당없음,NULL이면 값 누락 여부 확인
...,...,...,...,...,...,...
121,122,전임교원_남,faculty_count,집계형 수치: 0은 없음/미발생으로 유지,0 유지: 집계값의 실제 0으로 해석,"NULL이면 데이터 품질 문제, 0은 유효값"
122,123,전임교원_여,faculty_count,집계형 수치: 0은 없음/미발생으로 유지,0 유지: 집계값의 실제 0으로 해석,"NULL이면 데이터 품질 문제, 0은 유효값"
123,124,비전임교원_계,faculty_count,집계형 수치: 0은 없음/미발생으로 유지,0 유지: 집계값의 실제 0으로 해석,"NULL이면 데이터 품질 문제, 0은 유효값"
124,125,비전임교원_남,faculty_count,집계형 수치: 0은 없음/미발생으로 유지,0 유지: 집계값의 실제 0으로 해석,"NULL이면 데이터 품질 문제, 0은 유효값"


## 9. 실질 결측 산출

,column,semantic_type,raw_null_n,raw_null_pct,structural_null_n,optional_reference_null_n,true_missing_n,true_missing_pct,zero_n,zero_pct,all_zero_column,risk_level,cleaning_action
0,대학원구분,conditional_category,22757,64.229,22757,0,0,0.000,0,NaN,False,low,구조적 해당없음/placeholder로 별도 표기
1,팩스번호,contact_reference_optional,4818,13.598,0,4818,0,0.000,0,NaN,False,low,참조정보 누락: 분석 변수에서는 제외 가능
2,전화번호,contact_reference_optional,1242,3.505,0,1242,0,0.000,0,NaN,False,low,참조정보 누락: 분석 변수에서는 제외 가능
3,홈페이지,contact_reference_optional,1013,2.859,0,1013,0,0.000,0,NaN,False,low,참조정보 누락: 분석 변수에서는 제외 가능
4,대계열,taxonomy_category,462,1.304,462,0,0,0.000,0,NaN,False,low,구조적 해당없음/placeholder로 별도 표기
5,중계열,taxonomy_category,462,1.304,462,0,0,0.000,0,NaN,False,low,구조적 해당없음/placeholder로 별도 표기
6,소계열,taxonomy_category,462,1.304,462,0,0,0.000,0,NaN,False,low,구조적 해당없음/placeholder로 별도 표기


### NULL 샘플 행

,null_column,null_interpretation,연도,학제,대학원구분,학교명,본분교,시도,학위과정,대계열,중계열,소계열,학과코드,학과명,학과수_전체,입학정원_학부_계,지원자_전체_계,재적생_전체_계
0,대학원구분,"비대학원 행이면 해당없음, 대학원 행이면 실질결측",2024,대학교,<NA>,국립강릉원주대학교,본교(제1캠퍼스),강원,대학과정,인문계열,언어ㆍ문학,국어ㆍ국문학,U01010200008,국어국문학과,1,29,111,152
1,대학원구분,"비대학원 행이면 해당없음, 대학원 행이면 실질결측",2024,대학교,<NA>,국립강릉원주대학교,본교(제1캠퍼스),강원,대학과정,인문계열,언어ㆍ문학,중국어ㆍ문학,U01010400016,중어중문학과,1,26,120,109
2,대학원구분,"비대학원 행이면 해당없음, 대학원 행이면 실질결측",2024,대학교,<NA>,국립강릉원주대학교,본교(제1캠퍼스),강원,대학과정,인문계열,언어ㆍ문학,영미어ㆍ문학,U01010600018,영어영문학과,1,32,116,162
3,대학원구분,"비대학원 행이면 해당없음, 대학원 행이면 실질결측",2024,대학교,<NA>,국립강릉원주대학교,본교(제1캠퍼스),강원,대학과정,인문계열,언어ㆍ문학,독일어ㆍ문학,U01010700003,독어독문학과,1,0,0,39
4,대학원구분,"비대학원 행이면 해당없음, 대학원 행이면 실질결측",2024,대학교,<NA>,국립강릉원주대학교,본교(제1캠퍼스),강원,대학과정,인문계열,인문과학,역사ㆍ고고학,U01020400006,사학과,1,21,102,115
5,대학원구분,"비대학원 행이면 해당없음, 대학원 행이면 실질결측",2024,대학교,<NA>,국립강릉원주대학교,본교(제1캠퍼스),강원,대학과정,인문계열,인문과학,국제지역학,U01020600060,일본학과,1,26,160,149
6,대학원구분,"비대학원 행이면 해당없음, 대학원 행이면 실질결측",2024,대학교,<NA>,국립강릉원주대학교,본교(제1캠퍼스),강원,대학과정,인문계열,인문과학,철학ㆍ윤리학,U01020700029,철학과,1,22,105,99
7,대학원구분,"비대학원 행이면 해당없음, 대학원 행이면 실질결측",2024,대학교,<NA>,국립강릉원주대학교,본교(제1캠퍼스),강원,대학과정,인문계열,인문과학,교양인문학,U01020800121,기타모집단위,0,0,0,0
8,대학원구분,"비대학원 행이면 해당없음, 대학원 행이면 실질결측",2024,대학교,<NA>,국립강릉원주대학교,본교(제1캠퍼스),강원,대학과정,사회계열,경영ㆍ경제,경영학,U02010100035,경영학과,1,30,251,192
9,대학원구분,"비대학원 행이면 해당없음, 대학원 행이면 실질결측",2024,대학교,<NA>,국립강릉원주대학교,본교(제1캠퍼스),강원,대학과정,사회계열,경영ㆍ경제,경영학,U02010100048,관광경영학과,1,43,224,255


### 행 단위 클리닝 요약

,check,row_count,interpretation
0,non_graduate_rows_with_null_graduate_type,22757,대학원구분 해당없음: NULL을 '해당없음_비대학원'으로 분석용 보정
1,graduate_rows_with_null_graduate_type,0,대학원구분 실질결측
2,no_department_placeholder_rows,462,"학과명/계열별 평균·분포에서는 제외, 총합 분석에서는 소속학과없음 항목으로 별도 집계"
3,no_department_placeholder_rows_with_nonzero_me...,461,placeholder 행의 비0 수치 존재 여부. 현재는 교원 계열 수치가 배정되어...
4,taxonomy_true_missing_rows,0,실제 학과 행의 계열 실질결측


### `소속학과없음` placeholder에 배정된 비0 수치

,column,placeholder_total,nonzero_rows,semantic_type,handling_note
3,비전임교원_계,8234,328,faculty_count,"학과별 분포/평균에서는 제외, 총합에서는 소속학과없음 교원 항목으로 별도 집계"
4,비전임교원_남,4752,300,faculty_count,"학과별 분포/평균에서는 제외, 총합에서는 소속학과없음 교원 항목으로 별도 집계"
5,비전임교원_여,3482,254,faculty_count,"학과별 분포/평균에서는 제외, 총합에서는 소속학과없음 교원 항목으로 별도 집계"
0,전임교원_계,459,323,faculty_count,"학과별 분포/평균에서는 제외, 총합에서는 소속학과없음 교원 항목으로 별도 집계"
1,전임교원_남,384,290,faculty_count,"학과별 분포/평균에서는 제외, 총합에서는 소속학과없음 교원 항목으로 별도 집계"
2,전임교원_여,75,53,faculty_count,"학과별 분포/평균에서는 제외, 총합에서는 소속학과없음 교원 항목으로 별도 집계"


In [9]:

degree_cleaning_summary = (
    df_cleaning_flags.groupby(["학위과정", "학제"], dropna=False)
    .agg(
        rows=("학교명", "count"),
        graduate_type_null=("대학원구분", lambda s: int(s.isna().sum())),
        no_department_placeholder=("is_no_department_placeholder", "sum"),
        nonzero_placeholder=("measure_nonzero_n", lambda s: int(((s > 0) & df_cleaning_flags.loc[s.index, "is_no_department_placeholder"]).sum())),
    )
    .reset_index()
    .sort_values(["학위과정", "rows"], ascending=[True, False])
)

taxonomy_missing_by_course = (
    df.assign(
        is_no_department_placeholder=is_no_department_placeholder,
        taxonomy_any_null=df[TAXONOMY_COLS].isna().any(axis=1),
        taxonomy_true_missing=df[TAXONOMY_COLS].isna().any(axis=1) & ~is_no_department_placeholder,
    )
    .groupby(["학위과정", "학제"], dropna=False)
    .agg(
        rows=("학교명", "count"),
        taxonomy_any_null=("taxonomy_any_null", "sum"),
        structural_placeholder_null=("is_no_department_placeholder", "sum"),
        taxonomy_true_missing=("taxonomy_true_missing", "sum"),
    )
    .reset_index()
    .sort_values(["taxonomy_true_missing", "taxonomy_any_null"], ascending=False)
)

cleaning_recommendations = pd.DataFrame(
    [
        {
            "target": "대학원구분",
            "recommendation": "분석용 컬럼 `대학원구분_clean`을 사용한다. 비대학원 행의 NULL은 `해당없음_비대학원`, 대학원 행의 NULL만 실질결측으로 본다.",
            "risk_if_ignored": "원본 NULL 기준으로 dropna하면 전체 행의 64.2%를 잘못 제거한다.",
        },
        {
            "target": "대계열/중계열/소계열",
            "recommendation": "`소속학과없음` placeholder 행의 NULL은 구조적 결측으로 본다. 학과별 평균/분포에서는 제외하고, 교원 총합에서는 `소속학과없음` 교원 항목으로 별도 집계한다.",
            "risk_if_ignored": "학과가 아닌 462개 placeholder 행이 계열미상으로 들어가 학과 분포를 왜곡한다. 반대로 무조건 삭제하면 일부 교원 총합이 빠진다.",
        },
        {
            "target": "입학정원/모집인원/지원자/입학자",
            "recommendation": "경쟁형 수치로 분류하고 0을 유지한다. 경쟁률/충원율 같은 파생비율을 만들 때만 분모 0 결과를 NULL 처리한다.",
            "risk_if_ignored": "0을 결측으로 바꾸면 미모집/미지원 학과가 사라지고 경쟁 강도가 과대평가된다.",
        },
        {
            "target": "재적생/외국인유학생/졸업자/교원",
            "recommendation": "집계형 수치로 분류하고 0을 유지한다.",
            "risk_if_ignored": "0명 관측을 결측 처리하면 평균과 총합이 왜곡된다.",
        },
        {
            "target": "전화번호/팩스번호/홈페이지",
            "recommendation": "참조정보 누락으로 분리한다. 본 분석의 핵심 결측으로 보지 않는다.",
            "risk_if_ignored": "연락처가 없는 학교/학과를 불필요하게 분석에서 제외하게 된다.",
        },
        {
            "target": "시간강사_계/남/여",
            "recommendation": "원본에는 보존하지만 분석용 `df`에서는 제거 완료. 산출 통계와 결측/0 audit에는 포함하지 않는다.",
            "risk_if_ignored": "전 행 0인 컬럼이 교원 관련 분석과 결측/0 audit을 불필요하게 흔든다.",
        },
    ]
)

degree_cleaning_summary.to_csv(TABLE_DIR / "18_degree_cleaning_summary.csv", index=False, encoding=OUTPUT_ENCODING)
taxonomy_missing_by_course.to_csv(TABLE_DIR / "19_taxonomy_missing_by_course.csv", index=False, encoding=OUTPUT_ENCODING)
cleaning_recommendations.to_csv(TABLE_DIR / "20_cleaning_recommendations.csv", index=False, encoding=OUTPUT_ENCODING)

display(Markdown("## 10. 샘플/집단별 클리닝 점검"))
display(Markdown("### 학위과정 x 학제별 해당없음/placeholder 분포"))
display(degree_cleaning_summary)
display(Markdown("### 계열 결측의 구조적 여부"))
display(taxonomy_missing_by_course)
display(Markdown("### 클리닝 권고"))
display(cleaning_recommendations)


/tmp/ipykernel_138988/577677036.py:14: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df.assign(
/tmp/ipykernel_138988/577677036.py:14: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df.assign(
/tmp/ipykernel_138988/577677036.py:14: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df.assign(


## 10. 샘플/집단별 클리닝 점검

### 학위과정 x 학제별 해당없음/placeholder 분포

,학위과정,학제,rows,graduate_type_null,no_department_placeholder,nonzero_placeholder
3,대학과정,대학교,14823,14823,181,180
6,대학과정,사이버대학(대학),497,497,17,17
7,대학과정,산업대학,168,168,3,3
1,대학과정,교육대학,153,153,3,3
0,대학과정,각종대학(대학),36,36,3,3
4,대학과정,방송통신대학교,36,36,0,0
8,대학과정,원격대학(대학),12,12,1,1
2,대학과정,기술대학,5,5,0,0
5,대학과정,사내대학(대학),4,4,0,0
9,대학원과정,일반대학원,6980,0,20,20


### 계열 결측의 구조적 여부

,학위과정,학제,rows,taxonomy_any_null,structural_placeholder_null,taxonomy_true_missing
3,대학과정,대학교,14823,181,181,0
17,전문대학과정,전문대학,6545,119,119,0
11,대학원과정,특수대학원,4827,52,52,0
12,전문대학과정,기능대학,347,30,30,0
10,대학원과정,전문대학원,867,27,27,0
9,대학원과정,일반대학원,6980,20,20,0
6,대학과정,사이버대학(대학),497,17,17,0
0,대학과정,각종대학(대학),36,3,3,0
1,대학과정,교육대학,153,3,3,0
7,대학과정,산업대학,168,3,3,0


### 클리닝 권고

,target,recommendation,risk_if_ignored
0,대학원구분,분석용 컬럼 `대학원구분_clean`을 사용한다. 비대학원 행의 NULL은 `해당없...,원본 NULL 기준으로 dropna하면 전체 행의 64.2%를 잘못 제거한다.
1,대계열/중계열/소계열,`소속학과없음` placeholder 행의 NULL은 구조적 결측으로 본다. 학과별...,학과가 아닌 462개 placeholder 행이 계열미상으로 들어가 학과 분포를 왜...
2,입학정원/모집인원/지원자/입학자,경쟁형 수치로 분류하고 0을 유지한다. 경쟁률/충원율 같은 파생비율을 만들 때만 분...,0을 결측으로 바꾸면 미모집/미지원 학과가 사라지고 경쟁 강도가 과대평가된다.
3,재적생/외국인유학생/졸업자/교원,집계형 수치로 분류하고 0을 유지한다.,0명 관측을 결측 처리하면 평균과 총합이 왜곡된다.
4,전화번호/팩스번호/홈페이지,참조정보 누락으로 분리한다. 본 분석의 핵심 결측으로 보지 않는다.,연락처가 없는 학교/학과를 불필요하게 분석에서 제외하게 된다.
5,시간강사_계/남/여,원본에는 보존하지만 분석용 `df`에서는 제거 완료. 산출 통계와 결측/0 audi...,전 행 0인 컬럼이 교원 관련 분석과 결측/0 audit을 불필요하게 흔든다.


In [10]:

df_analysis_total = df.copy()
df_analysis_total["is_no_department_placeholder"] = is_no_department_placeholder
df_analysis_total["analysis_unit_type"] = np.where(
    is_no_department_placeholder,
    "소속학과없음_placeholder",
    "학과",
)
df_analysis_total["대학원구분_clean"] = np.where(
    is_graduate_course,
    df_analysis_total["대학원구분"].astype("string").fillna("실질결측_대학원구분"),
    df_analysis_total["대학원구분"].astype("string").fillna("해당없음_비대학원"),
)
for col in TAXONOMY_COLS:
    clean_col = f"{col}_clean"
    df_analysis_total[clean_col] = df_analysis_total[col].astype("string")
    df_analysis_total.loc[is_no_department_placeholder & df_analysis_total[col].isna(), clean_col] = "소속학과없음"
    df_analysis_total[clean_col] = df_analysis_total[clean_col].fillna("실질결측_계열미상")

df_department = df_analysis_total[~df_analysis_total["is_no_department_placeholder"]].copy()
df_placeholder = df_analysis_total[df_analysis_total["is_no_department_placeholder"]].copy()

FACULTY_COLS = [col for col in ["전임교원_계", "전임교원_남", "전임교원_여", "비전임교원_계", "비전임교원_남", "비전임교원_여"] if col in df_analysis_total.columns]
FACULTY_KEY_COLS = [
    "연도",
    "학제",
    "대학원구분_clean",
    "학교명",
    "본분교",
    "시도",
    "학위과정",
    "analysis_unit_type",
    "학과코드",
    "학과명",
    "대계열_clean",
    "중계열_clean",
    "소계열_clean",
]
FACULTY_KEY_COLS = [col for col in FACULTY_KEY_COLS if col in df_analysis_total.columns]
df_faculty_total = df_analysis_total[FACULTY_KEY_COLS + FACULTY_COLS].copy()

analysis_frame_manifest = pd.DataFrame(
    [
        {
            "frame_name": "df_analysis_total",
            "path": str((ANALYSIS_FRAME_DIR / "P2_G0_analysis_total.csv").relative_to(BASE_DIR)),
            "rows": len(df_analysis_total),
            "columns": df_analysis_total.shape[1],
            "unit": "원자료 행 전체. 소속학과없음 placeholder 포함",
            "use_for": "전체/학교/시도 총합, 교원 총합, 원자료 재현",
            "caution": "학과별 평균/분포에는 placeholder가 섞이므로 바로 쓰지 않음",
        },
        {
            "frame_name": "df_department",
            "path": str((ANALYSIS_FRAME_DIR / "P2_G0_department_analysis.csv").relative_to(BASE_DIR)),
            "rows": len(df_department),
            "columns": df_department.shape[1],
            "unit": "실제 학과 행. 소속학과없음 placeholder 제외",
            "use_for": "학과별 분포, 대계열/중계열/소계열별 평균, 경쟁률/충원율 EDA",
            "caution": "교원 총합 분석에서는 placeholder 교원 수치가 빠질 수 있음",
        },
        {
            "frame_name": "df_placeholder",
            "path": str((ANALYSIS_FRAME_DIR / "P2_G0_placeholder_rows.csv").relative_to(BASE_DIR)),
            "rows": len(df_placeholder),
            "columns": df_placeholder.shape[1],
            "unit": "소속학과없음 placeholder 행",
            "use_for": "학과 분포 제외 근거, 교원 총합 보정",
            "caution": "교원 수치가 배정된 행이 있으므로 무조건 삭제하지 않음",
        },
        {
            "frame_name": "df_faculty_total",
            "path": str((ANALYSIS_FRAME_DIR / "P2_G0_faculty_total_frame.csv").relative_to(BASE_DIR)),
            "rows": len(df_faculty_total),
            "columns": df_faculty_total.shape[1],
            "unit": "교원 총합용 행. placeholder 포함",
            "use_for": "학교/시도/학위과정별 전임/비전임 교원 합계",
            "caution": "시간강사 컬럼은 분석용 df에서 제거됨",
        },
    ]
)

analysis_frame_checks = pd.DataFrame(
    [
        {
            "check": "time_lecturer_removed_from_analysis_total",
            "result": not any(col in df_analysis_total.columns for col in DROPPED_ANALYSIS_COLUMNS),
            "value": ", ".join([col for col in DROPPED_ANALYSIS_COLUMNS if col in df_analysis_total.columns]) or "removed",
        },
        {
            "check": "department_rows_exclude_placeholder",
            "result": bool(~df_department["is_no_department_placeholder"].any()),
            "value": int(df_department["is_no_department_placeholder"].sum()),
        },
        {
            "check": "placeholder_rows_count",
            "result": len(df_placeholder) == int(is_no_department_placeholder.sum()),
            "value": len(df_placeholder),
        },
        {
            "check": "department_plus_placeholder_equals_total",
            "result": len(df_department) + len(df_placeholder) == len(df_analysis_total),
            "value": f"{len(df_department)} + {len(df_placeholder)} = {len(df_analysis_total)}",
        },
        {
            "check": "graduate_type_true_missing_in_graduate_course",
            "result": int((is_graduate_course & df['대학원구분'].isna()).sum()) == 0,
            "value": int((is_graduate_course & df['대학원구분'].isna()).sum()),
        },
        {
            "check": "taxonomy_true_missing_in_department_rows",
            "result": int(df_department[TAXONOMY_COLS].isna().any(axis=1).sum()) == 0,
            "value": int(df_department[TAXONOMY_COLS].isna().any(axis=1).sum()),
        },
    ]
)

df_analysis_total.to_csv(ANALYSIS_FRAME_DIR / "P2_G0_analysis_total.csv", index=False, encoding=OUTPUT_ENCODING)
df_department.to_csv(ANALYSIS_FRAME_DIR / "P2_G0_department_analysis.csv", index=False, encoding=OUTPUT_ENCODING)
df_placeholder.to_csv(ANALYSIS_FRAME_DIR / "P2_G0_placeholder_rows.csv", index=False, encoding=OUTPUT_ENCODING)
df_faculty_total.to_csv(ANALYSIS_FRAME_DIR / "P2_G0_faculty_total_frame.csv", index=False, encoding=OUTPUT_ENCODING)
analysis_frame_manifest.to_csv(TABLE_DIR / "23_analysis_frame_manifest.csv", index=False, encoding=OUTPUT_ENCODING)
analysis_frame_checks.to_csv(TABLE_DIR / "24_analysis_frame_checks.csv", index=False, encoding=OUTPUT_ENCODING)

display(Markdown("## 11. 식별된 클리닝 정책 반영: 분석 프레임 확정"))
display(analysis_frame_manifest)
display(Markdown("### 분석 프레임 검증"))
display(analysis_frame_checks)


/tmp/ipykernel_138988/2439977455.py:91: DeprecationWarning: Bitwise inversion '~' on bool is deprecated. This returns the bitwise inversion of the underlying int object and is usually not what you expect from negating a bool. Use the 'not' operator for boolean negation or ~int(x) if you really want the bitwise inversion of the underlying int.
  "result": bool(~df_department["is_no_department_placeholder"].any()),


## 11. 식별된 클리닝 정책 반영: 분석 프레임 확정

,frame_name,path,rows,columns,unit,use_for,caution
0,df_analysis_total,final/eda/P2_G0/analysis_frames/P2_G0_analysis...,35431,132,원자료 행 전체. 소속학과없음 placeholder 포함,"전체/학교/시도 총합, 교원 총합, 원자료 재현",학과별 평균/분포에는 placeholder가 섞이므로 바로 쓰지 않음
1,df_department,final/eda/P2_G0/analysis_frames/P2_G0_departme...,34969,132,실제 학과 행. 소속학과없음 placeholder 제외,"학과별 분포, 대계열/중계열/소계열별 평균, 경쟁률/충원율 EDA",교원 총합 분석에서는 placeholder 교원 수치가 빠질 수 있음
2,df_placeholder,final/eda/P2_G0/analysis_frames/P2_G0_placehol...,462,132,소속학과없음 placeholder 행,"학과 분포 제외 근거, 교원 총합 보정",교원 수치가 배정된 행이 있으므로 무조건 삭제하지 않음
3,df_faculty_total,final/eda/P2_G0/analysis_frames/P2_G0_faculty_...,35431,19,교원 총합용 행. placeholder 포함,학교/시도/학위과정별 전임/비전임 교원 합계,시간강사 컬럼은 분석용 df에서 제거됨


### 분석 프레임 검증

,check,result,value
0,time_lecturer_removed_from_analysis_total,True,removed
1,department_rows_exclude_placeholder,True,0
2,placeholder_rows_count,True,462
3,department_plus_placeholder_equals_total,True,34969 + 462 = 35431
4,graduate_type_true_missing_in_graduate_course,True,0
5,taxonomy_true_missing_in_department_rows,True,0


In [11]:

final_department = df_department.copy()
final_total = df_analysis_total.copy()
final_placeholder = df_placeholder.copy()
final_faculty_total = df_faculty_total.copy()

final_data_files = [
    {
        "dataset": "P2_G0_final_department_cleaned",
        "frame_name": "final_department",
        "path": FINAL_DATA_DIR / "P2_G0_final_department_cleaned.csv",
        "rows": len(final_department),
        "columns": final_department.shape[1],
        "primary": True,
        "unit": "실제 학과 행",
        "intended_use": "학과별 분포, 계열별 평균, 경쟁률/충원율 등 기본 EDA",
    },
    {
        "dataset": "P2_G0_final_total_with_placeholder",
        "frame_name": "final_total",
        "path": FINAL_DATA_DIR / "P2_G0_final_total_with_placeholder.csv",
        "rows": len(final_total),
        "columns": final_total.shape[1],
        "primary": False,
        "unit": "원자료 행 전체",
        "intended_use": "전체 총합 재현, placeholder 포함 원자료 감사",
    },
    {
        "dataset": "P2_G0_final_faculty_total",
        "frame_name": "final_faculty_total",
        "path": FINAL_DATA_DIR / "P2_G0_final_faculty_total.csv",
        "rows": len(final_faculty_total),
        "columns": final_faculty_total.shape[1],
        "primary": False,
        "unit": "교원 총합용 행",
        "intended_use": "소속학과없음 교원 수치를 포함한 학교/시도/학위과정별 교원 총합",
    },
    {
        "dataset": "P2_G0_final_placeholder_rows",
        "frame_name": "final_placeholder",
        "path": FINAL_DATA_DIR / "P2_G0_final_placeholder_rows.csv",
        "rows": len(final_placeholder),
        "columns": final_placeholder.shape[1],
        "primary": False,
        "unit": "소속학과없음 placeholder 행",
        "intended_use": "학과 분석 제외 근거와 교원 총합 보정 감사",
    },
]

for entry in final_data_files:
    frame_obj = {
        "final_department": final_department,
        "final_total": final_total,
        "final_faculty_total": final_faculty_total,
        "final_placeholder": final_placeholder,
    }[entry["frame_name"]]
    frame_obj.to_csv(entry["path"], index=False, encoding=OUTPUT_ENCODING)

final_data_manifest = pd.DataFrame(
    [
        {
            **{key: value for key, value in entry.items() if key != "path"},
            "path": str(entry["path"].relative_to(BASE_DIR)),
            "encoding": OUTPUT_ENCODING,
        }
        for entry in final_data_files
    ]
)

final_data_checks = pd.DataFrame(
    [
        {
            "check": "final_department_exists",
            "result": (FINAL_DATA_DIR / "P2_G0_final_department_cleaned.csv").exists(),
            "value": str(FINAL_DATA_DIR / "P2_G0_final_department_cleaned.csv"),
        },
        {
            "check": "time_lecturer_removed_from_all_final_data",
            "result": not any(
                col in frame.columns
                for frame in [final_department, final_total, final_faculty_total, final_placeholder]
                for col in DROPPED_ANALYSIS_COLUMNS
            ),
            "value": "removed",
        },
        {
            "check": "final_department_has_no_placeholder_rows",
            "result": bool(~final_department["is_no_department_placeholder"].any()),
            "value": int(final_department["is_no_department_placeholder"].sum()),
        },
        {
            "check": "final_placeholder_row_count",
            "result": len(final_placeholder) == 462,
            "value": len(final_placeholder),
        },
        {
            "check": "final_total_reconciles_department_and_placeholder",
            "result": len(final_department) + len(final_placeholder) == len(final_total),
            "value": f"{len(final_department)} + {len(final_placeholder)} = {len(final_total)}",
        },
        {
            "check": "final_department_taxonomy_missing",
            "result": int(final_department[TAXONOMY_COLS].isna().any(axis=1).sum()) == 0,
            "value": int(final_department[TAXONOMY_COLS].isna().any(axis=1).sum()),
        },
        {
            "check": "final_department_graduate_type_clean_missing",
            "result": int(final_department["대학원구분_clean"].isna().sum()) == 0,
            "value": int(final_department["대학원구분_clean"].isna().sum()),
        },
    ]
)

final_column_schema = pd.DataFrame(
    [
        {
            "dataset": dataset_name,
            "column_order": idx,
            "column": col,
            "dtype": str(frame[col].dtype),
            "non_null_n": int(frame[col].notna().sum()),
            "missing_n": int(frame[col].isna().sum()),
            "unique_n": int(frame[col].nunique(dropna=True)),
        }
        for dataset_name, frame in [
            ("P2_G0_final_department_cleaned", final_department),
            ("P2_G0_final_total_with_placeholder", final_total),
            ("P2_G0_final_faculty_total", final_faculty_total),
            ("P2_G0_final_placeholder_rows", final_placeholder),
        ]
        for idx, col in enumerate(frame.columns, start=1)
    ]
)

final_data_manifest.to_csv(FINAL_DATA_DIR / "P2_G0_final_data_manifest.csv", index=False, encoding=OUTPUT_ENCODING)
final_data_checks.to_csv(FINAL_DATA_DIR / "P2_G0_final_data_checks.csv", index=False, encoding=OUTPUT_ENCODING)
final_column_schema.to_csv(FINAL_DATA_DIR / "P2_G0_final_column_schema.csv", index=False, encoding=OUTPUT_ENCODING)

display(Markdown("## 13. 최종 클리닝 데이터 저장"))
display(final_data_manifest)
display(Markdown("### 최종 데이터 검증"))
display(final_data_checks)


/tmp/ipykernel_138988/4255292380.py:87: DeprecationWarning: Bitwise inversion '~' on bool is deprecated. This returns the bitwise inversion of the underlying int object and is usually not what you expect from negating a bool. Use the 'not' operator for boolean negation or ~int(x) if you really want the bitwise inversion of the underlying int.
  "result": bool(~final_department["is_no_department_placeholder"].any()),


## 13. 최종 클리닝 데이터 저장

,dataset,frame_name,rows,columns,primary,unit,intended_use,path,encoding
0,P2_G0_final_department_cleaned,final_department,34969,132,True,실제 학과 행,"학과별 분포, 계열별 평균, 경쟁률/충원율 등 기본 EDA",final/eda/P2_G0/final_data/P2_G0_final_departm...,utf-8-sig
1,P2_G0_final_total_with_placeholder,final_total,35431,132,False,원자료 행 전체,"전체 총합 재현, placeholder 포함 원자료 감사",final/eda/P2_G0/final_data/P2_G0_final_total_w...,utf-8-sig
2,P2_G0_final_faculty_total,final_faculty_total,35431,19,False,교원 총합용 행,소속학과없음 교원 수치를 포함한 학교/시도/학위과정별 교원 총합,final/eda/P2_G0/final_data/P2_G0_final_faculty...,utf-8-sig
3,P2_G0_final_placeholder_rows,final_placeholder,462,132,False,소속학과없음 placeholder 행,학과 분석 제외 근거와 교원 총합 보정 감사,final/eda/P2_G0/final_data/P2_G0_final_placeho...,utf-8-sig


### 최종 데이터 검증

,check,result,value
0,final_department_exists,True,/home/sieg/projects-wsl/SBS_dataScience/workbo...
1,time_lecturer_removed_from_all_final_data,True,removed
2,final_department_has_no_placeholder_rows,True,0
3,final_placeholder_row_count,True,462
4,final_total_reconciles_department_and_placeholder,True,34969 + 462 = 35431
5,final_department_taxonomy_missing,True,0
6,final_department_graduate_type_clean_missing,True,0


In [12]:

roadmap = pd.DataFrame(
    [
        {
            "phase": "Phase 1",
            "title": "파생변수 정의",
            "notebook_target": "P2_G0.ipynb",
            "main_frame": "df_department",
            "tasks": "지원경쟁률, 입학률, 외국인유학생비중, 졸업자/재적생, 교원1인당 재적생 생성",
            "decision_needed": "분모 0일 때 파생비율 NULL 처리 원칙 확정",
            "output": "25_derived_metric_profile.csv",
        },
        {
            "phase": "Phase 2",
            "title": "분포/이상값 EDA",
            "notebook_target": "P2_G0.ipynb",
            "main_frame": "df_department",
            "tasks": "학위과정, 학제, 시도, 대계열별 분포와 상위 outlier 점검",
            "decision_needed": "상위 극단값을 원자료 유지/별도 주석/분석 제외 중 선택",
            "output": "figures + outlier tables",
        },
        {
            "phase": "Phase 3",
            "title": "교원 총합 보정 EDA",
            "notebook_target": "P2_G0.ipynb",
            "main_frame": "df_faculty_total",
            "tasks": "소속학과없음 교원 수치를 포함한 학교/시도/학위과정별 교원 합계 산출",
            "decision_needed": "교원 지표는 학과별 평균과 총합을 분리 보고",
            "output": "faculty_total_summary.csv",
        },
        {
            "phase": "Phase 4",
            "title": "기사/보고서용 질문 후보",
            "notebook_target": "P2_G0.ipynb 또는 별도 hypothesis notebook",
            "main_frame": "df_department",
            "tasks": "지역별 경쟁 강도, 계열별 외국인유학생 집중, 교원 대비 학생 부담 등 질문 후보 생성",
            "decision_needed": "보도/분석 질문 2~3개 선택",
            "output": "claim_evidence_matrix.csv",
        },
        {
            "phase": "Phase 5",
            "title": "최종 분석 프레임 동결",
            "notebook_target": "final notebook",
            "main_frame": "frozen analysis CSVs",
            "tasks": "검증된 CSV만 로드하는 clean notebook 작성",
            "decision_needed": "최종 채택 지표와 제외 기준 확정",
            "output": "clean report notebook + figures",
        },
    ]
)
roadmap.to_csv(TABLE_DIR / "25_p2_g0_eda_roadmap.csv", index=False, encoding=OUTPUT_ENCODING)

display(Markdown("## 12. 앞으로의 EDA 로드맵"))
display(roadmap)


## 12. 앞으로의 EDA 로드맵

,phase,title,notebook_target,main_frame,tasks,decision_needed,output
0,Phase 1,파생변수 정의,P2_G0.ipynb,df_department,"지원경쟁률, 입학률, 외국인유학생비중, 졸업자/재적생, 교원1인당 재적생 생성",분모 0일 때 파생비율 NULL 처리 원칙 확정,25_derived_metric_profile.csv
1,Phase 2,분포/이상값 EDA,P2_G0.ipynb,df_department,"학위과정, 학제, 시도, 대계열별 분포와 상위 outlier 점검",상위 극단값을 원자료 유지/별도 주석/분석 제외 중 선택,figures + outlier tables
2,Phase 3,교원 총합 보정 EDA,P2_G0.ipynb,df_faculty_total,소속학과없음 교원 수치를 포함한 학교/시도/학위과정별 교원 합계 산출,교원 지표는 학과별 평균과 총합을 분리 보고,faculty_total_summary.csv
3,Phase 4,기사/보고서용 질문 후보,P2_G0.ipynb 또는 별도 hypothesis notebook,df_department,"지역별 경쟁 강도, 계열별 외국인유학생 집중, 교원 대비 학생 부담 등 질문 후보 생성",보도/분석 질문 2~3개 선택,claim_evidence_matrix.csv
4,Phase 5,최종 분석 프레임 동결,final notebook,frozen analysis CSVs,검증된 CSV만 로드하는 clean notebook 작성,최종 채택 지표와 제외 기준 확정,clean report notebook + figures


In [13]:

saved_outputs = pd.DataFrame(
    [
        {"path": str(path.relative_to(BASE_DIR)), "bytes": path.stat().st_size}
        for path in [*sorted(TABLE_DIR.glob("*.csv")), *sorted(ANALYSIS_FRAME_DIR.glob("*.csv")), *sorted(FINAL_DATA_DIR.glob("*.csv"))]
    ]
)

display(Markdown("## 14. 이번 단계 산출물"))
display(saved_outputs)
print(f"df shape = {df.shape}")
print(f"df_department shape = {df_department.shape}")
print(f"analysis frames saved to: {ANALYSIS_FRAME_DIR}")
print(f"final data saved to: {FINAL_DATA_DIR}")
print(f"tables saved to: {TABLE_DIR}")


## 14. 이번 단계 산출물

,path,bytes
0,final/eda/P2_G0/tables/00_workbook_profile.csv,148
1,final/eda/P2_G0/tables/01_header_preview_rows_...,6367
2,final/eda/P2_G0/tables/02_dataframe_overview.csv,677
3,final/eda/P2_G0/tables/03_numeric_conversion_a...,4053
4,final/eda/P2_G0/tables/04_dataframe_head30.csv,19512
5,final/eda/P2_G0/tables/05_column_profile.csv,11130
6,final/eda/P2_G0/tables/06_numeric_descriptive_...,17040
7,final/eda/P2_G0/tables/07_numeric_descriptive_...,11170
8,final/eda/P2_G0/tables/08_categorical_top_coun...,6691
9,final/eda/P2_G0/tables/09_candidate_key_audit.csv,604


df shape = (35431, 126)
df_department shape = (34969, 132)
analysis frames saved to: /home/sieg/projects-wsl/SBS_dataScience/workbook/p2/p2_2/final/eda/P2_G0/analysis_frames
final data saved to: /home/sieg/projects-wsl/SBS_dataScience/workbook/p2/p2_2/final/eda/P2_G0/final_data
tables saved to: /home/sieg/projects-wsl/SBS_dataScience/workbook/p2/p2_2/final/eda/P2_G0/tables


In [14]:

# P2_G2_FINAL_EDA_POISSON_CELL_V1
from pathlib import Path

from IPython.display import Markdown, display
import matplotlib.pyplot as plt
from matplotlib import font_manager
import numpy as np
import pandas as pd
from scipy.stats import poisson

try:
    BASE_DIR
except NameError:
    BASE_DIR = Path("/home/sieg/projects-wsl/SBS_dataScience/workbook/p2/p2_2")

OUTPUT_ENCODING = "utf-8-sig"
FINAL_CSV = BASE_DIR / "final/final_data/P2_G2_메인_입결_A_취업진학.CSV"
FINAL_EDA_DIR = BASE_DIR / "final/eda/P2_G0"
FINAL_TABLE_DIR = FINAL_EDA_DIR / "tables"
FINAL_FIGURE_DIR = FINAL_EDA_DIR / "figures"
FINAL_TABLE_DIR.mkdir(parents=True, exist_ok=True)
FINAL_FIGURE_DIR.mkdir(parents=True, exist_ok=True)

available_fonts = {font.name for font in font_manager.fontManager.ttflist}
for font_name in ["NanumGothic", "Noto Sans CJK KR", "Noto Sans KR", "Malgun Gothic", "DejaVu Sans"]:
    if font_name in available_fonts:
        plt.rcParams["font.family"] = font_name
        break
plt.rcParams["axes.unicode_minus"] = False

final_df = pd.read_csv(FINAL_CSV, encoding=OUTPUT_ENCODING, low_memory=False)

overview = pd.DataFrame(
    [
        {"metric": "rows", "value": len(final_df), "note": "최종 주간/비사이버/실제 학과 분석 행"},
        {"metric": "columns", "value": final_df.shape[1], "note": "최종 클리닝 후 컬럼 수"},
        {"metric": "schools", "value": final_df["학교명"].nunique(dropna=True), "note": "학교명 unique"},
        {"metric": "department_codes", "value": final_df["학과코드"].nunique(dropna=True), "note": "학과코드 unique"},
        {"metric": "large_major_groups", "value": final_df["대계열"].nunique(dropna=True), "note": "대계열 unique"},
        {"metric": "missing_cells", "value": int(final_df.isna().sum().sum()), "note": "남은 결측 셀 수"},
    ]
)

missing_profile = (
    final_df.isna()
    .sum()
    .rename("missing_n")
    .reset_index()
    .rename(columns={"index": "column"})
    .assign(missing_rate=lambda x: x["missing_n"] / len(final_df))
    .query("missing_n > 0")
    .sort_values(["missing_n", "column"], ascending=[False, True])
)

preferred_category_cols = [
    "학제",
    "대학원구분",
    "본분교",
    "시도",
    "시군구",
    "설립",
    "학위과정",
    "대계열",
    "중계열",
    "소계열",
    "대학원구분_clean",
]
category_cols = [col for col in preferred_category_cols if col in final_df.columns]

poisson_rows = []
for col in category_cols:
    values = final_df[col].astype("string").fillna("<<NULL>>").str.strip()
    counts = values.value_counts(dropna=False)
    if len(counts) < 2:
        continue
    lam = len(values) / len(counts)
    poisson_cutoff_5pct = int(poisson.ppf(0.05, lam))
    for category, count in counts.items():
        count = int(count)
        lower_tail_p = float(poisson.cdf(count, lam))
        poisson_rows.append(
            {
                "column": col,
                "category": category,
                "count": count,
                "share": count / len(values),
                "category_n": len(counts),
                "poisson_equal_share_lambda": lam,
                "poisson_5pct_count_cutoff": poisson_cutoff_5pct,
                "poisson_lower_tail_p": lower_tail_p,
                "low_frequency_flag": bool((lower_tail_p <= 0.05) or (count <= 5)),
                "interpretation": "동일 컬럼 내 범주가 균등하게 발생한다고 본 포아송 기준의 저빈도 경고",
            }
        )

poisson_category_profile = pd.DataFrame(poisson_rows).sort_values(
    ["low_frequency_flag", "poisson_lower_tail_p", "count"],
    ascending=[False, True, True],
)

low_frequency_summary = (
    poisson_category_profile.groupby("column", as_index=False)
    .agg(
        category_n=("category", "nunique"),
        low_frequency_category_n=("low_frequency_flag", "sum"),
        min_count=("count", "min"),
        median_count=("count", "median"),
        max_count=("count", "max"),
        poisson_lambda=("poisson_equal_share_lambda", "first"),
        poisson_5pct_cutoff=("poisson_5pct_count_cutoff", "first"),
    )
    .sort_values(["low_frequency_category_n", "category_n"], ascending=[False, False])
)

numeric_cols = final_df.select_dtypes(include=[np.number]).columns.tolist()
sparse_rows = []
for col in numeric_cols:
    series = final_df[col].dropna()
    if series.empty:
        continue
    zero_n = int((series == 0).sum())
    nonzero_n = int((series != 0).sum())
    zero_rate = zero_n / len(series)
    lam = float(series.mean())
    expected_nonzero_n = float(len(series) * (1 - poisson.pmf(0, lam))) if lam >= 0 else np.nan
    if zero_rate >= 0.90 or nonzero_n <= 100:
        sparse_rows.append(
            {
                "column": col,
                "non_null_n": len(series),
                "zero_n": zero_n,
                "nonzero_n": nonzero_n,
                "zero_rate": zero_rate,
                "mean_lambda": lam,
                "poisson_expected_nonzero_n": expected_nonzero_n,
                "nonzero_gap_observed_minus_poisson": nonzero_n - expected_nonzero_n,
                "note": "0을 결측으로 바꾸지 말고 희소 카운트 변수로 해석",
            }
        )

sparse_numeric_profile = pd.DataFrame(sparse_rows).sort_values(
    ["zero_rate", "nonzero_n"], ascending=[False, True]
)

overview.to_csv(FINAL_TABLE_DIR / "26_p2_g2_final_overview.csv", index=False, encoding=OUTPUT_ENCODING)
missing_profile.to_csv(FINAL_TABLE_DIR / "27_p2_g2_final_missing_profile.csv", index=False, encoding=OUTPUT_ENCODING)
poisson_category_profile.to_csv(FINAL_TABLE_DIR / "28_p2_g2_poisson_low_frequency_categories.csv", index=False, encoding=OUTPUT_ENCODING)
low_frequency_summary.to_csv(FINAL_TABLE_DIR / "29_p2_g2_poisson_low_frequency_summary.csv", index=False, encoding=OUTPUT_ENCODING)
sparse_numeric_profile.to_csv(FINAL_TABLE_DIR / "30_p2_g2_sparse_numeric_poisson_profile.csv", index=False, encoding=OUTPUT_ENCODING)

plot_rare = (
    poisson_category_profile[poisson_category_profile["low_frequency_flag"]]
    .sort_values(["poisson_lower_tail_p", "count"])
    .head(30)
    .copy()
)

rare_fig_path = FINAL_FIGURE_DIR / "26_p2_g2_poisson_low_frequency_categories.png"
if not plot_rare.empty:
    plot_rare["label"] = plot_rare["column"] + " = " + plot_rare["category"].astype(str)
    plot_rare = plot_rare.sort_values("count")
    y = np.arange(len(plot_rare))
    fig_height = max(7, 0.38 * len(plot_rare))
    fig, ax = plt.subplots(figsize=(12, fig_height))
    ax.barh(y, plot_rare["count"].clip(lower=1), color="#4C78A8", alpha=0.82, label="관측 빈도")
    ax.scatter(
        plot_rare["poisson_5pct_count_cutoff"].clip(lower=1),
        y,
        color="#D62728",
        s=34,
        label="포아송 5% 하한 컷오프",
        zorder=3,
    )
    ax.scatter(
        plot_rare["poisson_equal_share_lambda"].clip(lower=1),
        y,
        color="#2CA02C",
        s=28,
        label="동일비중 기대빈도 λ",
        zorder=3,
    )
    ax.set_yticks(y)
    ax.set_yticklabels(plot_rare["label"])
    ax.set_xscale("log")
    ax.set_xlabel("빈도 로그 스케일")
    ax.set_title("최종 CSV 저빈도 범주: 포아송 동일비중 기준")
    ax.grid(axis="x", alpha=0.25)
    ax.legend(loc="lower right")
    for idx, count in enumerate(plot_rare["count"]):
        ax.text(max(count, 1) * 1.05, idx, f"{count:,}", va="center", fontsize=8)
    fig.tight_layout()
    fig.savefig(rare_fig_path, dpi=180, bbox_inches="tight")
    plt.show()

sparse_fig_path = FINAL_FIGURE_DIR / "27_p2_g2_sparse_numeric_poisson_check.png"
if not sparse_numeric_profile.empty:
    plot_sparse = sparse_numeric_profile.head(25).sort_values("nonzero_n")
    y = np.arange(len(plot_sparse))
    fig_height = max(6, 0.34 * len(plot_sparse))
    fig, ax = plt.subplots(figsize=(12, fig_height))
    ax.barh(y, plot_sparse["nonzero_n"].clip(lower=1), color="#F58518", alpha=0.80, label="관측 non-zero 행 수")
    ax.scatter(
        plot_sparse["poisson_expected_nonzero_n"].clip(lower=1),
        y,
        color="#6F4E7C",
        s=36,
        label="포아송 평균 기준 기대 non-zero 행 수",
        zorder=3,
    )
    ax.set_yticks(y)
    ax.set_yticklabels(plot_sparse["column"])
    ax.set_xscale("log")
    ax.set_xlabel("행 수 로그 스케일")
    ax.set_title("희소 수치 컬럼 점검: 0이 많은 카운트 변수")
    ax.grid(axis="x", alpha=0.25)
    ax.legend(loc="lower right")
    fig.tight_layout()
    fig.savefig(sparse_fig_path, dpi=180, bbox_inches="tight")
    plt.show()

display(Markdown("## 15. 최종 CSV 단일셀 EDA + 포아송 저빈도 점검"))
display(Markdown(f"- 분석 파일: `{FINAL_CSV}`"))
display(Markdown(f"- 저장 표: `{FINAL_TABLE_DIR}`"))
display(Markdown(f"- 저장 그림: `{FINAL_FIGURE_DIR}`"))
display(overview)
display(Markdown("### 남은 결측"))
display(missing_profile if not missing_profile.empty else pd.DataFrame([{"message": "남은 결측 없음"}]))
display(Markdown("### 포아송 저빈도 범주 요약"))
display(low_frequency_summary)
display(Markdown("### 포아송 기준 저빈도 범주 상위 30"))
display(plot_rare)
display(Markdown("### 희소 수치 컬럼 포아송 점검"))
display(sparse_numeric_profile.head(30))

print(f"final_df shape = {final_df.shape}")
print(f"low-frequency categories flagged = {int(poisson_category_profile['low_frequency_flag'].sum())}")
print(f"rare category figure = {rare_fig_path}")
print(f"sparse numeric figure = {sparse_fig_path}")


## 15. 최종 CSV 단일셀 EDA + 포아송 저빈도 점검

- 분석 파일: `/home/sieg/projects-wsl/SBS_dataScience/workbook/p2/p2_2/final/final_data/P2_G2_메인_입결_A_취업진학.CSV`

- 저장 표: `/home/sieg/projects-wsl/SBS_dataScience/workbook/p2/p2_2/final/eda/P2_G0/tables`

- 저장 그림: `/home/sieg/projects-wsl/SBS_dataScience/workbook/p2/p2_2/final/eda/P2_G0/figures`

,metric,value,note
0,rows,15727,최종 주간/비사이버/실제 학과 분석 행
1,columns,120,최종 클리닝 후 컬럼 수
2,schools,587,학교명 unique
3,department_codes,8296,학과코드 unique
4,large_major_groups,7,대계열 unique
5,missing_cells,8260,남은 결측 셀 수


...

### 포아송 기준 저빈도 범주 상위 30

,column,category,count,share,category_n,poisson_equal_share_lambda,poisson_5pct_count_cutoff,poisson_lower_tail_p,low_frequency_flag,interpretation,label
9,학제,사내대학(대학),1,0.000,10,"1,572.700",1508,0.000,True,동일 컬럼 내 범주가 균등하게 발생한다고 본 포아송 기준의 저빈도 경고,학제 = 사내대학(대학)
8,학제,사내대학(전문),2,0.000,10,"1,572.700",1508,0.000,True,동일 컬럼 내 범주가 균등하게 발생한다고 본 포아송 기준의 저빈도 경고,학제 = 사내대학(전문)
17,본분교,본교(제4캠퍼스),27,0.002,5,"3,145.400",3053,0.000,True,동일 컬럼 내 범주가 균등하게 발생한다고 본 포아송 기준의 저빈도 경고,본분교 = 본교(제4캠퍼스)
7,학제,각종대학(대학),29,0.002,10,"1,572.700",1508,0.000,True,동일 컬럼 내 범주가 균등하게 발생한다고 본 포아송 기준의 저빈도 경고,학제 = 각종대학(대학)
229,중계열,특수교육,76,0.005,36,436.861,403,0.000,True,동일 컬럼 내 범주가 균등하게 발생한다고 본 포아송 기준의 저빈도 경고,중계열 = 특수교육
183,설립,기타,87,0.006,7,"2,246.714",2169,0.000,True,동일 컬럼 내 범주가 균등하게 발생한다고 본 포아송 기준의 저빈도 경고,설립 = 기타
34,시도,울산,92,0.006,17,925.118,875,0.000,True,동일 컬럼 내 범주가 균등하게 발생한다고 본 포아송 기준의 저빈도 경고,시도 = 울산
228,중계열,연극ㆍ영화,94,0.006,36,436.861,403,0.000,True,동일 컬럼 내 범주가 균등하게 발생한다고 본 포아송 기준의 저빈도 경고,중계열 = 연극ㆍ영화
33,시도,제주,95,0.006,17,925.118,875,0.000,True,동일 컬럼 내 범주가 균등하게 발생한다고 본 포아송 기준의 저빈도 경고,시도 = 제주
182,설립,특별법국립,97,0.006,7,"2,246.714",2169,0.000,True,동일 컬럼 내 범주가 균등하게 발생한다고 본 포아송 기준의 저빈도 경고,설립 = 특별법국립


### 희소 수치 컬럼 포아송 점검

,column,non_null_n,zero_n,nonzero_n,zero_rate,mean_lambda,poisson_expected_nonzero_n,nonzero_gap_observed_minus_poisson,note
0,정원외_입학정원_학부,15727,15642,85,0.995,0.107,"1,598.772","-1,513.772",0을 결측으로 바꾸지 말고 희소 카운트 변수로 해석
4,유예생_전체_남,15727,14760,967,0.939,0.188,"2,695.645","-1,728.645",0을 결측으로 바꾸지 말고 희소 카운트 변수로 해석
3,정원외_입학자_박사_여,15727,14520,1207,0.923,0.240,"3,352.179","-2,145.179",0을 결측으로 바꾸지 말고 희소 카운트 변수로 해석
2,정원외_입학자_박사_남,15727,14484,1243,0.921,0.242,"3,374.977","-2,131.977",0을 결측으로 바꾸지 말고 희소 카운트 변수로 해석
5,유예생_전체_여,15727,14455,1272,0.919,0.356,"4,715.701","-3,443.701",0을 결측으로 바꾸지 말고 희소 카운트 변수로 해석
1,정원외_입학자_석사_남,15727,14306,1421,0.910,0.392,"5,095.481","-3,674.481",0을 결측으로 바꾸지 말고 희소 카운트 변수로 해석


final_df shape = (15727, 120)
low-frequency categories flagged = 244
rare category figure = /home/sieg/projects-wsl/SBS_dataScience/workbook/p2/p2_2/final/eda/P2_G0/figures/26_p2_g2_poisson_low_frequency_categories.png
sparse numeric figure = /home/sieg/projects-wsl/SBS_dataScience/workbook/p2/p2_2/final/eda/P2_G0/figures/27_p2_g2_sparse_numeric_poisson_check.png
